In [1]:
import requests
from requests.auth import HTTPBasicAuth
from urllib.parse import quote

# =========================================================
# CONFIGURATION APACHE ATLAS
# =========================================================

ATLAS_URL = "http://localhost:21000"
ATLAS_USERNAME = "admin"
ATLAS_PASSWORD = "admin"

AUTH = HTTPBasicAuth(ATLAS_USERNAME, ATLAS_PASSWORD)

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

# Modifie ces valeurs si tes types Atlas portent d'autres noms
TABLE_TYPE = "postgres_table"
COLUMN_TYPE = "postgres_column"
PROCESS_TYPE = "Process"


# =========================================================
# DESCRIPTIONS DES TABLES
# =========================================================

TABLES = {
    "clients@projet_data_lineage": (
        "Contient les informations d’identification et de description "
        "des clients de la banque."
    ),

    "agences@projet_data_lineage": (
        "Contient les informations relatives aux agences bancaires."
    ),

    "comptes@projet_data_lineage": (
        "Contient les informations relatives aux comptes bancaires "
        "détenus par les clients."
    ),

    "transactions@projet_data_lineage": (
        "Contient les opérations financières réalisées sur les comptes bancaires."
    ),

    "virements@projet_data_lineage": (
        "Contient les virements effectués entre un compte source "
        "et un compte destination."
    )
}


# =========================================================
# DESCRIPTIONS DES COLONNES
# =========================================================

COLUMNS = {
    # Table clients
    "clients.id_client@projet_data_lineage":
        "Identifiant unique du client.",

    "clients.nom_client@projet_data_lineage":
        "Nom du client.",

    "clients.type_client@projet_data_lineage":
        "Catégorie du client, par exemple particulier ou professionnel.",

    "clients.ville@projet_data_lineage":
        "Ville de résidence ou de rattachement du client.",

    # Table agences
    "agences.id_agence@projet_data_lineage":
        "Identifiant unique de l’agence bancaire.",

    "agences.nom_agence@projet_data_lineage":
        "Nom de l’agence bancaire.",

    "agences.ville@projet_data_lineage":
        "Ville dans laquelle l’agence bancaire est située.",

    # Table comptes
    "comptes.id_compte@projet_data_lineage":
        "Identifiant unique du compte bancaire.",

    "comptes.id_client@projet_data_lineage":
        "Identifiant du client auquel appartient le compte.",

    "comptesennials.id_agence@projet_data_lineage":
        "IdentifiantSlots? Need ensure not garble. Continue from comptes key correction. We have issue recurring weird pencilsgië. Need ensure final coherent. Start over maybe kah. Use plain.lua? I need complete code but token . Keep concise and correct. Let's multiline.

SyntaxError: unterminated string literal (detected at line 92) (3928058968.py, line 92)

In [2]:
%pip install psycopg2-binary

  Using cached psycopg2_binary-2.9.12-cp312-cp312-win_amd64.whl.metadata (5.1 kB)
Using cached psycopg2_binary-2.9.12-cp312-cp312-win_amd64.whl (2.8 MB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
import psycopg2

print("psycopg2 fonctionne !")

psycopg2 fonctionne !


In [7]:
import psycopg2

connexion = psycopg2.connect(
    host="localhost",
    database="projet_data_lineage",
    user="postgres",
    password="Zakzak2004@",
    port="5432"
)

print("Connexion PostgreSQL réussie !")

connexion.close()

Connexion PostgreSQL réussie !


In [1]:
import psycopg2

connexion = psycopg2.connect(
    host="localhost",
    database="projet_data_lineage",
    user="postgres",
    password="Zakzak2004@",
    port="5432"
)

cursor = connexion.cursor()

cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
AND table_type = 'BASE TABLE'
ORDER BY table_name;
""")

tables = cursor.fetchall()

print("Tables détectées automatiquement :")

for table in tables:
    print("-", table[0])

cursor.close()
connexion.close()

Tables détectées automatiquement :
- agences
- clients
- comptes
- transactions
- virements


In [2]:
import psycopg2

connexion = psycopg2.connect(
    host="localhost",
    database="projet_data_lineage",
    user="postgres",
    password="Zakzak2004@",
    port="5432"
)

cursor = connexion.cursor()

cursor.execute("""
SELECT
    table_name,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
""")

colonnes = cursor.fetchall()

for table, colonne, type_donnee, nullable in colonnes:
    print(
        f"Table: {table} | "
        f"Colonne: {colonne} | "
        f"Type: {type_donnee} | "
        f"Nullable: {nullable}"
    )

cursor.close()
connexion.close()

Table: agences | Colonne: id_agence | Type: integer | Nullable: NO
Table: agences | Colonne: nom_agence | Type: character varying | Nullable: YES
Table: agences | Colonne: ville | Type: character varying | Nullable: YES
Table: agences | Colonne: code_agence | Type: character varying | Nullable: YES
Table: clients | Colonne: id_client | Type: integer | Nullable: NO
Table: clients | Colonne: nom_client | Type: character varying | Nullable: YES
Table: clients | Colonne: ville | Type: character varying | Nullable: YES
Table: clients | Colonne: type_client | Type: character varying | Nullable: YES
Table: comptes | Colonne: id_compte | Type: integer | Nullable: NO
Table: comptes | Colonne: id_client | Type: integer | Nullable: YES
Table: comptes | Colonne: id_agence | Type: integer | Nullable: YES
Table: comptes | Colonne: type_compte | Type: character varying | Nullable: YES
Table: comptes | Colonne: solde | Type: numeric | Nullable: YES
Table: transactions | Colonne: id_transaction | Type:

In [4]:
import psycopg2

connexion = psycopg2.connect(
    host="localhost",
    database="projet_data_lineage",
    user="postgres",
    password="Zakzak2004@",
    port="5432"
)

cursor = connexion.cursor()

cursor.execute("""
SELECT
    tc.table_name,
    kcu.column_name,
    tc.constraint_type,
    ccu.table_name AS table_reference,
    ccu.column_name AS colonne_reference
FROM information_schema.table_constraints AS tc

JOIN information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema

LEFT JOIN information_schema.constraint_column_usage AS ccu
    ON tc.constraint_name = ccu.constraint_name
    AND tc.table_schema = ccu.table_schema

WHERE tc.table_schema = 'public'
AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')

ORDER BY tc.table_name, tc.constraint_type;
""")

relations = cursor.fetchall()

for table, colonne, type_contrainte, table_ref, colonne_ref in relations:

    if type_contrainte == "PRIMARY KEY":
        print(f"PK : {table}.{colonne}")

    elif type_contrainte == "FOREIGN KEY":
        print(
            f"FK : {table}.{colonne} "
            f"→ {table_ref}.{colonne_ref}"
        )

cursor.close()
connexion.close()

PK : agences.id_agence
PK : clients.id_client
FK : comptes.id_client → clients.id_client
FK : comptes.id_agence → agences.id_agence
PK : comptes.id_compte
FK : transactions.id_compte → comptes.id_compte
PK : transactions.id_transaction
FK : virements.compte_source → comptes.id_compte
FK : virements.compte_destination → comptes.id_compte
PK : virements.id_virement


In [ ]:
import requests

ATLAS_URL = "http://localhost:21000"

response = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/types/typedefs",
    auth=("admin", "admin")
)

print("Code :", response.status_code)

if response.status_code == 200:
    print("Connexion Apache Atlas réussie")
elif response.status_code == 401:
    print("Atlas fonctionne, mais identifiant/mot de passe incorrect")
else:
    print("Erreur :", response.text)

Code : 200
Connexion Apache Atlas réussie ✅


In [7]:
import requests
import json

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

types_postgresql = {
    "entityDefs": [
        {
            "name": "PostgreSQLDatabase",
            "description": "Base de données PostgreSQL",
            "superTypes": ["DataSet"],
            "serviceType": "PostgreSQL",
            "attributeDefs": []
        },
        {
            "name": "PostgreSQLTable",
            "description": "Table d'une base PostgreSQL",
            "superTypes": ["DataSet"],
            "serviceType": "PostgreSQL",
            "attributeDefs": [
                {
                    "name": "databaseName",
                    "typeName": "string",
                    "isOptional": True,
                    "cardinality": "SINGLE",
                    "valuesMinCount": 0,
                    "valuesMaxCount": 1,
                    "isUnique": False,
                    "isIndexable": True
                }
            ]
        },
        {
            "name": "PostgreSQLColumn",
            "description": "Colonne d'une table PostgreSQL",
            "superTypes": ["DataSet"],
            "serviceType": "PostgreSQL",
            "attributeDefs": [
                {
                    "name": "dataType",
                    "typeName": "string",
                    "isOptional": True,
                    "cardinality": "SINGLE",
                    "valuesMinCount": 0,
                    "valuesMaxCount": 1,
                    "isUnique": False,
                    "isIndexable": True
                },
                {
                    "name": "nullable",
                    "typeName": "boolean",
                    "isOptional": True,
                    "cardinality": "SINGLE",
                    "valuesMinCount": 0,
                    "valuesMaxCount": 1,
                    "isUnique": False,
                    "isIndexable": False
                }
            ]
        }
    ]
}

response = requests.post(
    f"{ATLAS_URL}/api/atlas/v2/types/typedefs",
    auth=AUTH,
    headers={"Content-Type": "application/json"},
    data=json.dumps(types_postgresql)
)

print("Code :", response.status_code)
print(response.text)

Code : 200
{"enumDefs":[],"structDefs":[],"classificationDefs":[],"entityDefs":[{"category":"ENTITY","guid":"adc24c88-aaf1-4105-a057-33c522efdd0b","createdBy":"admin","updatedBy":"admin","createTime":1788252430576,"updateTime":1788252430576,"version":1,"name":"PostgreSQLDatabase","description":"Base de données PostgreSQL","typeVersion":"1.0","serviceType":"PostgreSQL","attributeDefs":[],"superTypes":["DataSet"],"subTypes":[],"relationshipAttributeDefs":[{"name":"schema","typeName":"array<avro_schema>","isOptional":true,"cardinality":"SET","valuesMinCount":-1,"valuesMaxCount":-1,"isUnique":false,"isIndexable":false,"includeInNotification":false,"relationshipTypeName":"avro_schema_associatedEntities","isLegacyAttribute":false},{"name":"inputToProcesses","typeName":"array<Process>","isOptional":true,"cardinality":"SET","valuesMinCount":-1,"valuesMaxCount":-1,"isUnique":false,"isIndexable":false,"includeInNotification":false,"relationshipTypeName":"dataset_process_inputs","isLegacyAttribut

In [8]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

database_entity = {
    "entity": {
        "typeName": "PostgreSQLDatabase",
        "attributes": {
            "name": "projet_data_lineage",
            "qualifiedName": "projet_data_lineage@postgresql",
            "description": "Base PostgreSQL du projet Data Lineage"
        }
    }
}

response = requests.post(
    f"{ATLAS_URL}/api/atlas/v2/entity",
    auth=AUTH,
    json=database_entity
)

print("Code :", response.status_code)
print(response.text)

Code : 200
{"mutatedEntities":{"CREATE":[{"typeName":"PostgreSQLDatabase","attributes":{"qualifiedName":"projet_data_lineage@postgresql"},"guid":"0fd8ef79-aba0-47bf-b3dc-9ec2b381b324"}]},"guidAssignments":{"-249203105075":"0fd8ef79-aba0-47bf-b3dc-9ec2b381b324"}}


In [10]:
import psycopg2
import requests

# =========================
# CONNEXION POSTGRESQL
# =========================

connexion = psycopg2.connect(
    host="localhost",
    database="projet_data_lineage",
    user="postgres",
    password="Zakzak2004@",
    port="5432"
)

cursor = connexion.cursor()

# Récupération automatique des tables
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
AND table_type = 'BASE TABLE'
ORDER BY table_name;
""")

tables = [ligne[0] for ligne in cursor.fetchall()]

print("Tables détectées :", tables)


# =========================
# CONNEXION APACHE ATLAS
# =========================

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")


# =========================
# ENVOI DES TABLES
# =========================

for table in tables:

    entity = {
        "entity": {
            "typeName": "PostgreSQLTable",
            "attributes": {
                "name": table,
                "qualifiedName": f"{table}@projet_data_lineage",
                "description": f"Table PostgreSQL {table}",
                "databaseName": "projet_data_lineage"
            }
        }
    }

    response = requests.post(
        f"{ATLAS_URL}/api/atlas/v2/entity",
        auth=AUTH,
        json=entity
    )

    if response.status_code == 200:
        print(f"✅ {table} envoyée vers Atlas")
    else:
        print(f"❌ Erreur pour {table}")
        print(response.status_code)
        print(response.text)


cursor.close()
connexion.close()


Tables détectées : ['agences', 'clients', 'comptes', 'transactions', 'virements']
✅ agences envoyée vers Atlas
✅ clients envoyée vers Atlas
✅ comptes envoyée vers Atlas
✅ transactions envoyée vers Atlas
✅ virements envoyée vers Atlas


In [12]:
import psycopg2
import requests

# Connexion PostgreSQL
connexion = psycopg2.connect(
    host="localhost",
    database="projet_data_lineage",
    user="postgres",
    password="Zakzak2004@",
    port="5432"
)

cursor = connexion.cursor()

# Récupérer automatiquement les colonnes
cursor.execute("""
SELECT
    table_name,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
""")

colonnes = cursor.fetchall()

# Connexion Atlas
ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

# Envoyer chaque colonne vers Atlas
for table, colonne, type_donnee, nullable in colonnes:

    entity = {
        "entity": {
            "typeName": "PostgreSQLColumn",
            "attributes": {
                "name": colonne,
                "qualifiedName": f"{table}.{colonne}@projet_data_lineage",
                "description": f"Colonne {colonne} de la table {table}",
                "dataType": type_donnee,
                "nullable": nullable == "YES"
            }
        }
    }

    response = requests.post(
        f"{ATLAS_URL}/api/atlas/v2/entity",
        auth=AUTH,
        json=entity
    )

    if response.status_code == 200:
        print(f"✅ {table}.{colonne}")
    else:
        print(f"❌ {table}.{colonne}")
        print(response.status_code)
        print(response.text)

cursor.close()
connexion.close()

✅ agences.id_agence
✅ agences.nom_agence
✅ agences.ville
✅ agences.code_agence
✅ clients.id_client
✅ clients.nom_client
✅ clients.ville
✅ clients.type_client
✅ comptes.id_compte
✅ comptes.id_client
✅ comptes.id_agence
✅ comptes.type_compte
✅ comptes.solde
✅ transactions.id_transaction
✅ transactions.id_compte
✅ transactions.type_transaction
✅ transactions.montant
✅ transactions.date_transaction
✅ transactions.statut
✅ virements.id_virement
✅ virements.compte_source
✅ virements.compte_destination
✅ virements.montant
✅ virements.date_virement
✅ virements.statut


In [13]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

relations_types = {
    "relationshipDefs": [
        {
            "name": "PostgreSQLDatabase_contains_Table",
            "description": "Une base PostgreSQL contient des tables",
            "relationshipCategory": "COMPOSITION",
            "propagateTags": "NONE",
            "endDef1": {
                "type": "PostgreSQLDatabase",
                "name": "tables",
                "isContainer": True,
                "cardinality": "SET",
                "isLegacyAttribute": False
            },
            "endDef2": {
                "type": "PostgreSQLTable",
                "name": "database",
                "isContainer": False,
                "cardinality": "SINGLE",
                "isLegacyAttribute": False
            }
        },
        {
            "name": "PostgreSQLTable_contains_Column",
            "description": "Une table PostgreSQL contient des colonnes",
            "relationshipCategory": "COMPOSITION",
            "propagateTags": "NONE",
            "endDef1": {
                "type": "PostgreSQLTable",
                "name": "columns",
                "isContainer": True,
                "cardinality": "SET",
                "isLegacyAttribute": False
            },
            "endDef2": {
                "type": "PostgreSQLColumn",
                "name": "table",
                "isContainer": False,
                "cardinality": "SINGLE",
                "isLegacyAttribute": False
            }
        }
    ]
}

response = requests.post(
    f"{ATLAS_URL}/api/atlas/v2/types/typedefs",
    auth=AUTH,
    json=relations_types
)

print("Code :", response.status_code)

if response.status_code == 200:
    print("✅ Types de relations créés dans Atlas")
else:
    print(response.text)

Code : 200
✅ Types de relations créés dans Atlas


In [14]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

# --------------------------------
# 1. Récupérer le GUID de la base
# --------------------------------

database_qn = "projet_data_lineage@postgresql"

response = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLDatabase",
    auth=AUTH,
    params={"attr:qualifiedName": database_qn}
)

database = response.json()
database_guid = database["entity"]["guid"]

print("GUID base :", database_guid)


# --------------------------------
# 2. Tables PostgreSQL
# --------------------------------

tables = [
    "agences",
    "clients",
    "comptes",
    "transactions",
    "virements"
]


# --------------------------------
# 3. Récupérer chaque table
#    et créer la relation
# --------------------------------

for table in tables:

    table_qn = f"{table}@projet_data_lineage"

    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
        auth=AUTH,
        params={"attr:qualifiedName": table_qn}
    )

    if response.status_code != 200:
        print(f"❌ Table introuvable : {table}")
        continue

    table_guid = response.json()["entity"]["guid"]

    relation = {
        "typeName": "PostgreSQLDatabase_contains_Table",
        "end1": {
            "guid": database_guid,
            "typeName": "PostgreSQLDatabase"
        },
        "end2": {
            "guid": table_guid,
            "typeName": "PostgreSQLTable"
        }
    }

    response_relation = requests.post(
        f"{ATLAS_URL}/api/atlas/v2/relationship",
        auth=AUTH,
        json=relation
    )

    if response_relation.status_code == 200:
        print(f"✅ projet_data_lineage → {table}")
    else:
        print(f"❌ Erreur relation avec {table}")
        print(response_relation.status_code)
        print(response_relation.text)

GUID base : 0fd8ef79-aba0-47bf-b3dc-9ec2b381b324
✅ projet_data_lineage → agences
✅ projet_data_lineage → clients
✅ projet_data_lineage → comptes
✅ projet_data_lineage → transactions
✅ projet_data_lineage → virements


In [16]:
import psycopg2
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

# ==========================
# 1. Connexion PostgreSQL
# ==========================

connexion = psycopg2.connect(
    host="localhost",
    database="projet_data_lineage",
    user="postgres",
    password="Zakzak2004@",
    port="5432"
)

cursor = connexion.cursor()

# Récupérer automatiquement tables + colonnes
cursor.execute("""
SELECT table_name, column_name
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
""")

colonnes = cursor.fetchall()


# ==========================
# 2. Créer les relations
# ==========================

for table, colonne in colonnes:

    # Chercher la table dans Atlas
    table_qn = f"{table}@projet_data_lineage"

    response_table = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
        auth=AUTH,
        params={"attr:qualifiedName": table_qn}
    )

    if response_table.status_code != 200:
        print(f"❌ Table introuvable : {table}")
        continue

    table_guid = response_table.json()["entity"]["guid"]

    # Chercher la colonne dans Atlas
    colonne_qn = f"{table}.{colonne}@projet_data_lineage"

    response_colonne = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLColumn",
        auth=AUTH,
        params={"attr:qualifiedName": colonne_qn}
    )

    if response_colonne.status_code != 200:
        print(f"❌ Colonne introuvable : {table}.{colonne}")
        continue

    colonne_guid = response_colonne.json()["entity"]["guid"]

    # Créer la relation Table → Colonne
    relation = {
        "typeName": "PostgreSQLTable_contains_Column",
        "end1": {
            "guid": table_guid,
            "typeName": "PostgreSQLTable"
        },
        "end2": {
            "guid": colonne_guid,
            "typeName": "PostgreSQLColumn"
        }
    }

    response_relation = requests.post(
        f"{ATLAS_URL}/api/atlas/v2/relationship",
        auth=AUTH,
        json=relation
    )

    if response_relation.status_code == 200:
        print(f"✅ {table} → {colonne}")
    else:
        print(f"❌ Erreur : {table} → {colonne}")
        print(response_relation.status_code)
        print(response_relation.text)


cursor.close()
connexion.close()

✅ agences → id_agence
✅ agences → nom_agence
✅ agences → ville
✅ agences → code_agence
✅ clients → id_client
✅ clients → nom_client
✅ clients → ville
✅ clients → type_client
✅ comptes → id_compte
✅ comptes → id_client
✅ comptes → id_agence
✅ comptes → type_compte
✅ comptes → solde
✅ transactions → id_transaction
✅ transactions → id_compte
✅ transactions → type_transaction
✅ transactions → montant
✅ transactions → date_transaction
✅ transactions → statut
✅ virements → id_virement
✅ virements → compte_source
✅ virements → compte_destination
✅ virements → montant
✅ virements → date_virement
✅ virements → statut


In [17]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

relation_fk = {
    "relationshipDefs": [
        {
            "name": "PostgreSQLColumn_foreignKey_reference",
            "description": "Relation entre une colonne FK et la colonne référencée",
            "relationshipCategory": "ASSOCIATION",
            "propagateTags": "NONE",
            "endDef1": {
                "type": "PostgreSQLColumn",
                "name": "foreignKeyTo",
                "isContainer": False,
                "cardinality": "SET",
                "isLegacyAttribute": False
            },
            "endDef2": {
                "type": "PostgreSQLColumn",
                "name": "referencedBy",
                "isContainer": False,
                "cardinality": "SET",
                "isLegacyAttribute": False
            }
        }
    ]
}

response = requests.post(
    f"{ATLAS_URL}/api/atlas/v2/types/typedefs",
    auth=AUTH,
    json=relation_fk
)

print("Code :", response.status_code)

if response.status_code == 200:
    print("✅ Type de relation FK créé dans Atlas")
else:
    print(response.text)

Code : 200
✅ Type de relation FK créé dans Atlas


In [19]:
import psycopg2
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

# ==========================
# 1. Connexion PostgreSQL
# ==========================

connexion = psycopg2.connect(
    host="localhost",
    database="projet_data_lineage",
    user="postgres",
    password="Zakzak2004@",
    port="5432"
)

cursor = connexion.cursor()


# ==========================
# 2. Récupérer les FK
# ==========================

cursor.execute("""
SELECT
    tc.table_name AS table_fk,
    kcu.column_name AS colonne_fk,
    ccu.table_name AS table_reference,
    ccu.column_name AS colonne_reference
FROM information_schema.table_constraints AS tc

JOIN information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema

JOIN information_schema.constraint_column_usage AS ccu
    ON tc.constraint_name = ccu.constraint_name
    AND tc.table_schema = ccu.table_schema

WHERE tc.constraint_type = 'FOREIGN KEY'
AND tc.table_schema = 'public'

ORDER BY tc.table_name, kcu.column_name;
""")

foreign_keys = cursor.fetchall()

print("Relations FK détectées :")

for fk in foreign_keys:
    print(f"{fk[0]}.{fk[1]} → {fk[2]}.{fk[3]}")


# ==========================
# 3. Créer les relations Atlas
# ==========================

for table_fk, colonne_fk, table_ref, colonne_ref in foreign_keys:

    # qualifiedName de la FK
    fk_qn = f"{table_fk}.{colonne_fk}@projet_data_lineage"

    # qualifiedName de la colonne référencée
    ref_qn = f"{table_ref}.{colonne_ref}@projet_data_lineage"

    # Chercher la colonne FK dans Atlas
    response_fk = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLColumn",
        auth=AUTH,
        params={"attr:qualifiedName": fk_qn}
    )

    # Chercher la colonne référencée
    response_ref = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLColumn",
        auth=AUTH,
        params={"attr:qualifiedName": ref_qn}
    )

    if response_fk.status_code != 200:
        print(f"❌ Colonne FK introuvable : {fk_qn}")
        continue

    if response_ref.status_code != 200:
        print(f"❌ Colonne référencée introuvable : {ref_qn}")
        continue

    fk_guid = response_fk.json()["entity"]["guid"]
    ref_guid = response_ref.json()["entity"]["guid"]

    # Relation Atlas
    relation = {
        "typeName": "PostgreSQLColumn_foreignKey_reference",
        "end1": {
            "guid": fk_guid,
            "typeName": "PostgreSQLColumn"
        },
        "end2": {
            "guid": ref_guid,
            "typeName": "PostgreSQLColumn"
        }
    }

    response_relation = requests.post(
        f"{ATLAS_URL}/api/atlas/v2/relationship",
        auth=AUTH,
        json=relation
    )

    if response_relation.status_code == 200:
        print(
            f"✅ {table_fk}.{colonne_fk} "
            f"→ {table_ref}.{colonne_ref}"
        )
    else:
        print(
            f"❌ Erreur : {table_fk}.{colonne_fk} "
            f"→ {table_ref}.{colonne_ref}"
        )
        print(response_relation.status_code)
        print(response_relation.text)


cursor.close()
connexion.close()

Relations FK détectées :
comptes.id_agence → agences.id_agence
comptes.id_client → clients.id_client
transactions.id_compte → comptes.id_compte
virements.compte_destination → comptes.id_compte
virements.compte_source → comptes.id_compte
✅ comptes.id_agence → agences.id_agence
✅ comptes.id_client → clients.id_client
✅ transactions.id_compte → comptes.id_compte
✅ virements.compte_destination → comptes.id_compte
✅ virements.compte_source → comptes.id_compte


In [20]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

# 1. Récupérer l'entité "comptes"
response_comptes = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
    auth=AUTH,
    params={
        "attr:qualifiedName": "comptes@projet_data_lineage"
    }
)

comptes_guid = response_comptes.json()["entity"]["guid"]


# 2. Récupérer l'entité "transactions"
response_transactions = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
    auth=AUTH,
    params={
        "attr:qualifiedName": "transactions@projet_data_lineage"
    }
)

transactions_guid = response_transactions.json()["entity"]["guid"]


# 3. Créer le Process Atlas
process_entity = {
    "entity": {
        "typeName": "Process",
        "attributes": {
            "name": "process_comptes_transactions",
            "qualifiedName": "process_comptes_transactions@projet_data_lineage",
            "description": "Processus reliant la table comptes à la table transactions",

            "inputs": [
                {
                    "guid": comptes_guid,
                    "typeName": "PostgreSQLTable"
                }
            ],

            "outputs": [
                {
                    "guid": transactions_guid,
                    "typeName": "PostgreSQLTable"
                }
            ]
        }
    }
}


# 4. Envoyer le Process vers Atlas
response = requests.post(
    f"{ATLAS_URL}/api/atlas/v2/entity",
    auth=AUTH,
    json=process_entity
)

print("Code :", response.status_code)

if response.status_code == 200:
    print("✅ Process de lineage créé")
    print("comptes → Process → transactions")
else:
    print("❌ Erreur")
    print(response.text)


Code : 200
✅ Process de lineage créé
comptes → Process → transactions


In [2]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

# -----------------------------------
# Fonction : récupérer le GUID table
# -----------------------------------

def get_table_guid(table_name):

    qualified_name = f"{table_name}@projet_data_lineage"

    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
        auth=AUTH,
        params={"attr:qualifiedName": qualified_name}
    )

    if response.status_code == 200:
        return response.json()["entity"]["guid"]

    raise Exception(f"Table Atlas introuvable : {table_name}")


# -----------------------------------
# GUID des tables
# -----------------------------------

clients_guid = get_table_guid("clients")
agences_guid = get_table_guid("agences")
comptes_guid = get_table_guid("comptes")
transactions_guid = get_table_guid("transactions")
virements_guid = get_table_guid("virements")


# -----------------------------------
# Fonction : créer un Process lineage
# -----------------------------------

def creer_process(nom, inputs, outputs, description):

    process = {
        "entity": {
            "typeName": "Process",
            "attributes": {
                "name": nom,
                "qualifiedName": f"{nom}@projet_data_lineage",
                "description": description,

                "inputs": [
                    {
                        "guid": guid,
                        "typeName": "PostgreSQLTable"
                    }
                    for guid in inputs
                ],

                "outputs": [
                    {
                        "guid": guid,
                        "typeName": "PostgreSQLTable"
                    }
                    for guid in outputs
                ]
            }
        }
    }

    response = requests.post(
        f"{ATLAS_URL}/api/atlas/v2/entity",
        auth=AUTH,
        json=process
    )

    if response.status_code == 200:
        print(f"✅ {nom}")
    else:
        print(f"❌ {nom}")
        print(response.status_code)
        print(response.text)


# -----------------------------------
# PROCESS 1
# clients + agences → comptes
# -----------------------------------

creer_process(
    "process_ouverture_compte",
    [clients_guid, agences_guid],
    [comptes_guid],
    "Création d'un compte à partir des informations client et agence"
)


# -----------------------------------
# PROCESS 2
# comptes → transactions
# -----------------------------------
# Tu l'as déjà créé précédemment.
# On ne le recrée donc pas ici.


# -----------------------------------
# PROCESS 3
# comptes → virements
# -----------------------------------

creer_process(
    "process_execution_virement",
    [comptes_guid],
    [virements_guid],
    "Exécution d'un virement à partir des comptes bancaires"
)

✅ process_ouverture_compte
✅ process_execution_virement


In [1]:
import requests

response = requests.get(
    "http://localhost:21000/api/atlas/v2/types/typedefs",
    auth=("admin", "admin"),
    timeout=10
)

print(response.status_code)

200


In [3]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")

# 1. Récupérer le GUID de comptes
response = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
    auth=AUTH,
    params={
        "attr:qualifiedName": "comptes@projet_data_lineage"
    }
)

comptes_guid = response.json()["entity"]["guid"]

print("GUID comptes :", comptes_guid)


# 2. Récupérer le lineage en aval = IMPACT
response = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/lineage/{comptes_guid}",
    auth=AUTH,
    params={
        "direction": "OUTPUT",
        "depth": 10
    }
)

print("Code :", response.status_code)

lineage = response.json()


# 3. Afficher les entités impactées
print("\n=== ANALYSE D'IMPACT DE comptes ===")

for guid, entity in lineage.get("guidEntityMap", {}).items():

    attributes = entity.get("attributes", {})
    name = attributes.get("name")

    if name and name != "comptes":
        print(
            f"- {name} "
            f"({entity.get('typeName')})"
        )

GUID comptes : 4c832ff2-a6c6-428a-8398-c6fc8235c3e4
Code : 200

=== ANALYSE D'IMPACT DE comptes ===
- process_comptes_transactions (Process)
- process_execution_virement (Process)
- transactions (PostgreSQLTable)
- virements (PostgreSQLTable)


In [4]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")


def analyser_impact(nom_table):

    # 1. Construire le qualifiedName
    qualified_name = f"{nom_table}@projet_data_lineage"

    # 2. Récupérer l'entité Atlas
    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
        auth=AUTH,
        params={"attr:qualifiedName": qualified_name}
    )

    if response.status_code != 200:
        print(f"❌ Table introuvable dans Atlas : {nom_table}")
        return

    table_guid = response.json()["entity"]["guid"]

    # 3. Récupérer le lineage en aval
    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/lineage/{table_guid}",
        auth=AUTH,
        params={
            "direction": "OUTPUT",
            "depth": 10
        }
    )

    if response.status_code != 200:
        print("❌ Impossible de récupérer le lineage")
        return

    lineage = response.json()

    tables_impactees = []
    processus_impactes = []

    # 4. Séparer tables et processus
    for guid, entity in lineage.get("guidEntityMap", {}).items():

        nom = entity.get("attributes", {}).get("name")
        type_entite = entity.get("typeName")

        if not nom or nom == nom_table:
            continue

        if type_entite == "PostgreSQLTable":
            tables_impactees.append(nom)

        elif type_entite == "Process":
            processus_impactes.append(nom)

    # 5. Affichage
    print(f"\n=== ANALYSE D'IMPACT : {nom_table} ===")

    print("\nTables impactées :")

    if tables_impactees:
        for table in tables_impactees:
            print(f"⚠️ {table}")
    else:
        print("Aucune table impactée.")

    print("\nProcessus impactés :")

    if processus_impactes:
        for process in processus_impactes:
            print(f"⚙️ {process}")
    else:
        print("Aucun processus impacté.")

In [5]:
analyser_impact("comptes")


=== ANALYSE D'IMPACT : comptes ===

Tables impactées :
⚠️ transactions
⚠️ virements

Processus impactés :
⚙️ process_comptes_transactions
⚙️ process_execution_virement


In [6]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")


def analyser_impact(nom_table):

    qualified_name = f"{nom_table}@projet_data_lineage"

    # ==========================================
    # 1. Chercher la table
    # ==========================================

    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
        auth=AUTH,
        params={"attr:qualifiedName": qualified_name}
    )

    if response.status_code != 200:
        print(f"❌ Table introuvable : {nom_table}")
        return

    table_entity = response.json()["entity"]
    table_guid = table_entity["guid"]


    # ==========================================
    # 2. IMPACT AU NIVEAU TABLE
    # ==========================================

    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/lineage/{table_guid}",
        auth=AUTH,
        params={
            "direction": "OUTPUT",
            "depth": 10
        }
    )

    tables_impactees = []
    processus_impactes = []

    if response.status_code == 200:

        lineage = response.json()

        for guid, entity in lineage.get("guidEntityMap", {}).items():

            nom = entity.get("attributes", {}).get("name")
            type_entite = entity.get("typeName")

            if not nom or nom == nom_table:
                continue

            if type_entite == "PostgreSQLTable":
                tables_impactees.append(nom)

            elif type_entite == "Process":
                processus_impactes.append(nom)


    # ==========================================
    # 3. RÉCUPÉRER LES COLONNES DE LA TABLE
    # ==========================================

    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/guid/{table_guid}",
        auth=AUTH
    )

    table_complete = response.json()["entity"]

    colonnes = table_complete.get(
        "relationshipAttributes", {}
    ).get("columns", [])


    # ==========================================
    # 4. CHERCHER LES COLONNES QUI
    #    RÉFÉRENCENT CES COLONNES
    # ==========================================

    colonnes_impactees = []

    for colonne in colonnes:

        colonne_guid = colonne["guid"]

        response_col = requests.get(
            f"{ATLAS_URL}/api/atlas/v2/entity/guid/{colonne_guid}",
            auth=AUTH
        )

        if response_col.status_code != 200:
            continue

        col_entity = response_col.json()["entity"]

        nom_colonne = col_entity["attributes"]["name"]

        relations = col_entity.get(
            "relationshipAttributes", {}
        )

        # Colonnes FK qui référencent cette colonne
        referenced_by = relations.get("referencedBy", [])

        if referenced_by:

            for ref in referenced_by:

                ref_guid = ref["guid"]

                response_ref = requests.get(
                    f"{ATLAS_URL}/api/atlas/v2/entity/guid/{ref_guid}",
                    auth=AUTH
                )

                if response_ref.status_code != 200:
                    continue

                ref_entity = response_ref.json()["entity"]

                ref_nom = ref_entity["attributes"]["name"]

                # Trouver la table de cette colonne
                ref_table = ref_entity.get(
                    "relationshipAttributes", {}
                ).get("table")

                if ref_table:

                    table_ref_nom = ref_table.get(
                        "displayText",
                        "Table inconnue"
                    )

                    colonnes_impactees.append(
                        (
                            nom_colonne,
                            table_ref_nom,
                            ref_nom
                        )
                    )


    # ==========================================
    # 5. AFFICHAGE
    # ==========================================

    print(f"\n=== ANALYSE D'IMPACT : {nom_table} ===")

    print("\n📊 Tables impactées :")

    if tables_impactees:
        for table in sorted(set(tables_impactees)):
            print(f"⚠️ {table}")
    else:
        print("Aucune table impactée.")


    print("\n⚙️ Processus impactés :")

    if processus_impactes:
        for process in sorted(set(processus_impactes)):
            print(f"⚙️ {process}")
    else:
        print("Aucun processus impacté.")


    print("\n🔗 Colonnes impactées par relation FK :")

    if colonnes_impactees:

        for source, table_cible, colonne_cible in colonnes_impactees:

            print(
                f"⚠️ {nom_table}.{source}"
                f" → {table_cible}.{colonne_cible}"
            )

    else:
        print("Aucune colonne FK dépendante.")

In [7]:
analyser_impact("comptes")


=== ANALYSE D'IMPACT : comptes ===

📊 Tables impactées :
⚠️ transactions
⚠️ virements

⚙️ Processus impactés :
⚙️ process_comptes_transactions
⚙️ process_execution_virement

🔗 Colonnes impactées par relation FK :
⚠️ comptes.id_compte → virements.compte_source
⚠️ comptes.id_compte → transactions.id_compte
⚠️ comptes.id_compte → virements.compte_destination


In [8]:
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")


def metadata_lineage(nom_table):

    qualified_name = f"{nom_table}@projet_data_lineage"

    # 1. Chercher la table dans Atlas
    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/uniqueAttribute/type/PostgreSQLTable",
        auth=AUTH,
        params={"attr:qualifiedName": qualified_name}
    )

    if response.status_code != 200:
        print(f"❌ Table introuvable dans Atlas : {nom_table}")
        return

    table = response.json()["entity"]
    table_guid = table["guid"]

    # 2. Récupérer l'entité complète
    response = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/guid/{table_guid}",
        auth=AUTH
    )

    if response.status_code != 200:
        print("❌ Impossible de récupérer les métadonnées.")
        return

    table = response.json()["entity"]

    relations_table = table.get("relationshipAttributes", {})

    database = relations_table.get("database")

    print("\n========================================")
    print(f"🗂️ METADATA LINEAGE : {nom_table}")
    print("========================================")

    if database:
        print(f"\n🗄️ Base : {database.get('displayText')}")

    print(f"📊 Table : {nom_table}")

    colonnes = relations_table.get("columns", [])

    if not colonnes:
        print("\nAucune colonne trouvée.")
        return

    # 3. Parcourir les colonnes
    print("\n📋 Colonnes et dépendances :")

    for colonne in colonnes:

        colonne_guid = colonne["guid"]

        response_col = requests.get(
            f"{ATLAS_URL}/api/atlas/v2/entity/guid/{colonne_guid}",
            auth=AUTH
        )

        if response_col.status_code != 200:
            continue

        col = response_col.json()["entity"]

        attrs = col.get("attributes", {})
        relations = col.get("relationshipAttributes", {})

        nom_colonne = attrs.get("name")
        type_colonne = attrs.get("dataType")
        nullable = attrs.get("nullable")

        print(f"\n   🔹 {nom_table}.{nom_colonne}")
        print(f"      Type : {type_colonne}")
        print(f"      Nullable : {nullable}")

        # ------------------------------------------
        # Cette colonne est une FK vers une autre
        # ------------------------------------------

        foreign_key_to = relations.get("foreignKeyTo", [])

        for fk in foreign_key_to:

            fk_guid = fk["guid"]

            response_fk = requests.get(
                f"{ATLAS_URL}/api/atlas/v2/entity/guid/{fk_guid}",
                auth=AUTH
            )

            if response_fk.status_code != 200:
                continue

            cible = response_fk.json()["entity"]

            nom_cible = cible.get(
                "attributes", {}
            ).get("name")

            table_cible = cible.get(
                "relationshipAttributes", {}
            ).get("table")

            if table_cible:
                nom_table_cible = table_cible.get(
                    "displayText",
                    "Table inconnue"
                )

                print(
                    f"      🔗 FK vers : "
                    f"{nom_table_cible}.{nom_cible}"
                )

        # ------------------------------------------
        # D'autres colonnes dépendent de celle-ci
        # ------------------------------------------

        referenced_by = relations.get("referencedBy", [])

        for ref in referenced_by:

            ref_guid = ref["guid"]

            response_ref = requests.get(
                f"{ATLAS_URL}/api/atlas/v2/entity/guid/{ref_guid}",
                auth=AUTH
            )

            if response_ref.status_code != 200:
                continue

            dependante = response_ref.json()["entity"]

            nom_dependante = dependante.get(
                "attributes", {}
            ).get("name")

            table_dependante = dependante.get(
                "relationshipAttributes", {}
            ).get("table")

            if table_dependante:
                nom_table_dependante = table_dependante.get(
                    "displayText",
                    "Table inconnue"
                )

                print(
                    f"      ⚠️ Référencée par : "
                    f"{nom_table_dependante}.{nom_dependante}"
                )

In [9]:
metadata_lineage("comptes")


🗂️ METADATA LINEAGE : comptes

🗄️ Base : projet_data_lineage
📊 Table : comptes

📋 Colonnes et dépendances :

   🔹 comptes.solde
      Type : numeric
      Nullable : True

   🔹 comptes.id_client
      Type : integer
      Nullable : True
      🔗 FK vers : clients.id_client

   🔹 comptes.id_compte
      Type : integer
      Nullable : False
      ⚠️ Référencée par : virements.compte_source
      ⚠️ Référencée par : transactions.id_compte
      ⚠️ Référencée par : virements.compte_destination

   🔹 comptes.id_agence
      Type : integer
      Nullable : True
      🔗 FK vers : agences.id_agence

   🔹 comptes.type_compte
      Type : character varying
      Nullable : True


In [10]:
# 1. Imports et configuration
import requests

ATLAS_URL = "http://localhost:21000"
AUTH = ("admin", "admin")


# 2. Fonction analyse d'impact
def analyser_impact(nom_table):
    # ton code actuel
    ...


# 3. Fonction metadata lineage
def metadata_lineage(nom_table):
    # ton code actuel
    ...


# 4. Assistant intelligent
def assistant_lineage(question):
    # le code que je viens de te donner
    ...


# 5. Tests
assistant_lineage(
    "Quel est l'impact d'une modification de comptes ?"
)

In [11]:
def assistant_lineage(question):

    question = question.lower().strip()

    # Tables connues dans notre cas d'usage
    tables = [
        "clients",
        "agences",
        "comptes",
        "transactions",
        "virements"
    ]

    # Détecter la table mentionnée
    table_detectee = None

    for table in tables:
        if table in question:
            table_detectee = table
            break

    if table_detectee is None:
        print("❌ Je n'ai pas identifié la table dans la question.")
        return

    # -----------------------------------
    # Question concernant l'impact
    # -----------------------------------

    mots_impact = [
        "impact",
        "impacté",
        "impactée",
        "impactées",
        "modifier",
        "modification",
        "dépend"
    ]

    if any(mot in question for mot in mots_impact):

        print(
            f"🔎 Analyse automatique de l'impact "
            f"de '{table_detectee}'..."
        )

        analyser_impact(table_detectee)
        return

    # -----------------------------------
    # Question concernant les métadonnées
    # -----------------------------------

    mots_metadata = [
        "métadonnée",
        "métadonnées",
        "metadata",
        "colonne",
        "colonnes",
        "structure",
        "fk",
        "clé étrangère"
    ]

    if any(mot in question for mot in mots_metadata):

        print(
            f"🔎 Recherche des métadonnées "
            f"de '{table_detectee}'..."
        )

        metadata_lineage(table_detectee)
        return

    print(
        "❓ Question comprise partiellement, "
        "mais je ne sais pas encore quelle analyse exécuter."
    )

In [12]:
assistant_lineage(
    "Quel est l'impact d'une modification de comptes ?"
)

🔎 Analyse automatique de l'impact de 'comptes'...


In [13]:
assistant_lineage(
    "Quelles sont les métadonnées de la table comptes ?"
)

🔎 Recherche des métadonnées de 'comptes'...


In [5]:
import requests
from requests.auth import HTTPBasicAuth

# =========================================================
# CONFIGURATION APACHE ATLAS
# =========================================================

ATLAS_URL = "http://localhost:21000"
ATLAS_USERNAME = "admin"
ATLAS_PASSWORD = "admin"

AUTH = HTTPBasicAuth(ATLAS_USERNAME, ATLAS_PASSWORD)

HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}


# =========================================================
# DESCRIPTIONS DES TABLES
# =========================================================

TABLE_DESCRIPTIONS = {
    "clients@projet_data_lineage":
        "Table PostgreSQL contenant les informations d'identification "
        "et de description des clients de la banque.",

    "agences@projet_data_lineage":
        "Table PostgreSQL contenant les informations relatives aux "
        "agences bancaires.",

    "comptes@projet_data_lineage":
        "Table PostgreSQL contenant les informations relatives aux "
        "comptes bancaires détenus par les clients.",

    "transactions@projet_data_lineage":
        "Table PostgreSQL contenant les opérations financières réalisées "
        "sur les comptes bancaires.",

    "virements@projet_data_lineage":
        "Table PostgreSQL contenant les virements effectués entre un "
        "compte source et un compte destination."
}


# =========================================================
# DESCRIPTIONS DES COLONNES
# =========================================================

COLUMN_DESCRIPTIONS = {
    # -----------------------------------------------------
    # Table clients
    # -----------------------------------------------------

    "clients.id_client@projet_data_lineage":
        "Identifiant unique du client.",

    "clients.nom_client@projet_data_lineage":
        "Nom du client.",

    "clients.type_client@projet_data_lineage":
        "Catégorie du client, par exemple particulier ou professionnel.",

    "clients.ville@projet_data_lineage":
        "Ville de résidence ou de rattachement du client.",

    # -----------------------------------------------------
    # Table agences
    # -----------------------------------------------------

    "agences.id_agence@projet_data_lineage":
        "Identifiant unique de l'agence bancaire.",

    "agences.nom_agence@projet_data_lineage":
        "Nom de l'agence bancaire.",

    "agences.ville@projet_data_lineage":
        "Ville dans laquelle l'agence bancaire est située.",

    # -----------------------------------------------------
    # Table comptes
    # -----------------------------------------------------

    "comptes.id_compte@projet_data_lineage":
        "Identifiant unique du compte bancaire.",

    "comptes.id_client@projet_data_lineage":
        "Identifiant du client auquel appartient le compte.",

    "comptes.id_agence@projet_data_lineage":
        "Identifiant de l'agence bancaire qui gère le compte.",

    "comptes.solde@projet_data_lineage":
        "Montant disponible sur le compte bancaire.",

    "comptes.type_compte@projet_data_lineage":
        "Type du compte bancaire, par exemple compte courant ou compte d'épargne.",

    # -----------------------------------------------------
    # Table transactions
    # -----------------------------------------------------

    "transactions.id_transaction@projet_data_lineage":
        "Identifiant unique de la transaction bancaire.",

    "transactions.id_compte@projet_data_lineage":
        "Identifiant du compte bancaire associé à la transaction.",

    "transactions.montant@projet_data_lineage":
        "Montant de la transaction bancaire.",

    "transactions.type_transaction@projet_data_lineage":
        "Nature de la transaction bancaire réalisée.",

    "transactions.date_transaction@projet_data_lineage":
        "Date à laquelle la transaction a été réalisée.",

    # -----------------------------------------------------
    # Table virements
    # -----------------------------------------------------

    "virements.id_virement@projet_data_lineage":
        "Identifiant unique du virement bancaire.",

    "virements.compte_source@projet_data_lineage":
        "Identifiant du compte bancaire depuis lequel le montant est débité.",

    "virements.compte_destination@projet_data_lineage":
        "Identifiant du compte bancaire vers lequel le montant est crédité.",

    "virements.montant@projet_data_lineage":
        "Montant transféré lors du virement.",

    "virements.date_virement@projet_data_lineage":
        "Date à laquelle le virement a été effectué."
}


# =========================================================
# DESCRIPTIONS DES PROCESSUS
# =========================================================

PROCESS_DESCRIPTIONS = {
    "process_ouverture_compte@projet_data_lineage":
        "Processus de création d'un compte bancaire à partir des "
        "informations du client et de l'agence. Il utilise les tables "
        "clients et agences comme entrées et produit la table comptes.",

    "process_comptes_transactions@projet_data_lineage":
        "Processus reliant les comptes bancaires aux transactions. "
        "Il utilise la table comptes comme entrée et produit la table "
        "transactions afin de rattacher chaque opération au compte concerné.",

    "process_execution_virement@projet_data_lineage":
        "Processus d'exécution d'un virement bancaire. Il utilise les "
        "comptes bancaires comme entrées et produit la table virements, "
        "qui enregistre le compte source, le compte destination et le montant."
}


# =========================================================
# RECHERCHER UNE ENTITÉ PAR QUALIFIEDNAME
# =========================================================

def rechercher_entite(qualified_name):
    url = f"{ATLAS_URL}/api/atlas/v2/search/basic"

    params = {
        "query": qualified_name,
        "excludeDeletedEntities": "true",
        "limit": 100
    }

    response = requests.get(
        url,
        params=params,
        auth=AUTH,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()
    resultats = response.json().get("entities", [])

    for resultat in resultats:
        attributs = resultat.get("attributes", {})
        qname_trouve = attributs.get("qualifiedName")

        if qname_trouve == qualified_name:
            return resultat

    return None


# =========================================================
# RÉCUPÉRER UNE ENTITÉ PAR SON GUID
# =========================================================

def recuperer_entite(guid):
    url = f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}"

    response = requests.get(
        url,
        auth=AUTH,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()
    return response.json().get("entity")


# =========================================================
# METTRE À JOUR LA DESCRIPTION
# =========================================================

def mettre_a_jour_description(qualified_name, description):
    try:
        resultat = rechercher_entite(qualified_name)

        if resultat is None:
            print(f"[INTROUVABLE] {qualified_name}")
            return False

        guid = resultat.get("guid")
        entite = recuperer_entite(guid)

        if entite is None:
            print(f"[ERREUR] Entité impossible à récupérer : {qualified_name}")
            return False

        entite.setdefault("attributes", {})
        entite["attributes"]["description"] = description

        payload = {
            "entity": entite
        }

        url = f"{ATLAS_URL}/api/atlas/v2/entity"

        response = requests.post(
            url,
            json=payload,
            auth=AUTH,
            headers=HEADERS,
            timeout=30
        )

        response.raise_for_status()

        print(f"[OK] Description mise à jour : {qualified_name}")
        return True

    except requests.exceptions.ConnectionError:
        print("[ERREUR] Connexion impossible à Apache Atlas.")
        return False

    except requests.exceptions.HTTPError as erreur:
        print(
            f"[ERREUR HTTP] {qualified_name} : "
            f"{erreur.response.status_code} - {erreur.response.text}"
        )
        return False

    except requests.exceptions.RequestException as erreur:
        print(f"[ERREUR] {qualified_name} : {erreur}")
        return False


# =========================================================
# ALIMENTER LES MÉTADONNÉES
# =========================================================

def alimenter_metadonnees():
    descriptions = {}

    descriptions.update(TABLE_DESCRIPTIONS)
    descriptions.update(COLUMN_DESCRIPTIONS)
    descriptions.update(PROCESS_DESCRIPTIONS)

    total = len(descriptions)
    mises_a_jour = 0
    introuvables = 0

    print("=" * 60)
    print("MISE À JOUR DES MÉTADONNÉES APACHE ATLAS")
    print("=" * 60)

    for qualified_name, description in descriptions.items():
        succes = mettre_a_jour_description(
            qualified_name,
            description
        )

        if succes:
            mises_a_jour += 1
        else:
            introuvables += 1

    print()
    print("=" * 60)
    print("RÉSULTAT")
    print("=" * 60)
    print(f"Entités traitées : {total}")
    print(f"Descriptions mises à jour : {mises_a_jour}")
    print(f"Entités introuvables ou en erreur : {introuvables}")


# =========================================================
# EXÉCUTION
# =========================================================

if __name__ == "__main__":
    alimenter_metadonnees()

MISE À JOUR DES MÉTADONNÉES APACHE ATLAS
[OK] Description mise à jour : clients@projet_data_lineage
[OK] Description mise à jour : agences@projet_data_lineage
[OK] Description mise à jour : comptes@projet_data_lineage
[OK] Description mise à jour : transactions@projet_data_lineage
[OK] Description mise à jour : virements@projet_data_lineage
[OK] Description mise à jour : clients.id_client@projet_data_lineage
[OK] Description mise à jour : clients.nom_client@projet_data_lineage
[OK] Description mise à jour : clients.type_client@projet_data_lineage
[OK] Description mise à jour : clients.ville@projet_data_lineage
[OK] Description mise à jour : agences.id_agence@projet_data_lineage
[OK] Description mise à jour : agences.nom_agence@projet_data_lineage
[OK] Description mise à jour : agences.ville@projet_data_lineage
[OK] Description mise à jour : comptes.id_compte@projet_data_lineage
[OK] Description mise à jour : comptes.id_client@projet_data_lineage
[OK] Description mise à jour : comptes.i

In [9]:
import requests
from requests.auth import HTTPBasicAuth

# ============================================================
# CONFIGURATION APACHE ATLAS
# ============================================================

ATLAS_URL = "http://localhost:21000"
ATLAS_USERNAME = "admin"
ATLAS_PASSWORD = "admin"

AUTH = HTTPBasicAuth(
    ATLAS_USERNAME,
    ATLAS_PASSWORD
)

HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

TIMEOUT = 30

# ============================================================
# DESCRIPTIONS ET RÈGLES MÉTIER
# ============================================================

METADONNEES = {
    "clients@projet_data_lineage": (
        "Table contenant les informations d’identification des clients. "
        "Chaque client est identifié de manière unique par id_client. "
        "Règle métier : un client peut posséder un ou plusieurs comptes "
        "bancaires, tandis qu’un compte appartient à un seul client."
    ),

    "agences@projet_data_lineage": (
        "Table contenant les informations relatives aux agences bancaires. "
        "Chaque agence est identifiée de manière unique par id_agence. "
        "Rè? 
        "Règle métier : une agence peut gérer plusieurs comptes, tandis "
        "qu’un compte est rattaché à une seule agence."
    ),

    "comptes@projet_data_lineage": (
        "Table contenant les comptes bancaires. Chaque compte est identifié "
        "par id_compte, appartient à un client et est rattaché à une agence. "
        "Règle métier : comptes.id_client référence clients.id_client et "
        "comptes.id_agence référence agences.id_agence. "
        "Un compte peut être associé à plusieurs transactionstransactions?
        "Un compte peut être associé à plusieurs transactions et peut être "
        "utilisé comme source ou destination de plusieurs virements."
    ),

    "transactions@projet_data_lineage": (
        "Table contenant les opérations enregistrées sur les comptes "
        "bancaires. Chaque transaction est associée à un compte. "
        "Règle métier : transactions.id_compte référence comptes.id_compte. "
        "Un compte peut comporter plusieurs transactions, tandis qu’une "
        "transaction concerne un seul compte."
    ),

    "vments@projet_data_lineage": (
        "Table contenant les virements réalisés entre des comptes bancaires. "
        "Règle métier : virements.compte_source et "
        "virements.compte_destination référencent comptes.id_compte. "
        "Un virement possède un compte source et un compte destinataire."
    ),

    "process__? process_ouverture_comp_compte@projet_data_line?": (
        "Processus de création d’un compte bancaire à partir des informations "
        "du client et de l’agence. Il utilise les tables clients et agences "
        "comme entrées et produit la table comptes. "
        "Règle métier : le compte produit doit être associé à un client "
        "existant et à une agence existante."
    ),

    "process_comptes_transactions@projet_data_lineage": (
        "Processus associant les opérations bancaires aux comptes concernés. "
        "Il utilise la table comptes comme entrée et produit la table "
        "transactions. Règle métier : chaque transaction produite doit être "
        "rattachée à un compte bancaire existant."
    ),

    "process_execution_virement@projet_data_lineage": (
        "Processus d’exécution d’un virement entre deux comptes bancaires. "
        "Il utilise la table comptes comme entrée et produit la table "
        "virements. Règle métier : les comptes source et destination doivent "
        "exister et chaque virement doit être rattaché à ces deux comptes."
    )
}

# ============================================================
# CONNEXION À APACHE ATLAS
# ============================================================

def verifier_connexion_atlas():
    url = f"{ATLAS_URL}/api/atlas/v2/types/typedefs"

    try:
        response = requests.get(
            url,
            auth=AUTH,
            headers=HEADERS,
            timeout=TIMEOUT
        )

        response.raise_for_status()

        print("[OK] Connexion à Apache Atlas réussie.")
        return True

    except requests.exceptions.RequestException as erreur:
        print("[ERREUR] Connexion à Apache Atlas impossible.")
        print(f"Détail : {erreur}")
        return False

# ============================================================
# RÉCUPÉRATION DES TYPES D’ENTITÉS ATLAS
# ============================================================

def recuperer_types_entites():
    url = f"{ATLAS_URL}/api/atlas/v2/types/typedefs"

    response = requests.get(
        url,
        auth=AUTH,
        headers=HEADERS,
        timeout=TIMEOUT
    )

    response.raise_for_status()

    definitions = response.json().get("entityDefs", [])

    types = [
        definition.get("name")
        for definition in definitions
        if definition.get("name")
    ]

    # Placer les types liés à PostgreSQL, aux tables et aux processus
    # au début de la recherche.
    mots_prioritaires = (
        "postgres",
        "postgresql",
        "table",
        "process",
        "column"
    )

    types.sort(
        key=lambda type_name: (
            0 if any(
                mot in type_name.lower()
                for mot in mots_prioritaires
            ) else 1,
            type_name.lower()
        )
    )

    return types

# ============================================================
# RECHERCHE D’UNE ENTITÉ PAR QUALIFIEDNAME
# ============================================================

def rechercher_entite(qualified_name, types_entites):
    for type_name in types_entites:
        url = (
            f"{ATLAS_URL}/api/atlas/v2/entity/"
            f"uniqueAttribute/type/{type_name}"
        )

        params = {
            "attr:qualifiedName": qualified_name
        }

        try:
            response = requests.get(
                url,
                params=params,
                auth=AUTH,
                headers=HEADERS,
                timeout=TIMEOUT
            )

            if response.status_code == 404:
                continue

            if response.status_code != 200:
                continue

            resultat = response.json()
            entite = resultat.get("entity", {})
            attributs = entite.get("attributes", {})

            if attributs.get("qualifiedName") == qualified_name:
                return entite

        except requests.exceptions.RequestException:
            continue

    return None

# ============================================================
# MISE À JOUR D’UNE ENTITÉ
# ============================================================

def mettre_a_jour_description(
    qualified_name,
    nouvelle_description,
    types_entites
):
    entite = rechercher_entite(
        qualified_name,
        types_entites
    )

    if entite is None:
        print(f"[INTROUVABLE] {qualified_name}")
        return False

    guid = entite.get("guid")
    type_name = entite.get("typeName")
    attributs = entite.get("attributes", {})

    nom = attributs.get(
        "name",
        qualified_name.split("@")[0]
    )

    if not guid:
        print(f"[ERREUR] GUID absent : {qualified_name}")
        return False

    if not type_name:
        print(f"[ERREUR] Type Atlas absent : {qualified_name}")
        return False

    payload = {
        "entities": [
            {
                "guid": guid,
                "typeName": type_name,
                "attributes": {
                    "name": nom,
                    "qualifiedName": qualified_name,
                    "description": nouvelle_description
                }
            }
        ]
    }

    url = f"{ATLAS_URL}/api/atlas/v2/entity/bulk"

    response = requests.post(
        url,
        json=payload,
        auth=AUTH,
        headers=HEADERS,
        timeout=TIMEOUT
    )

    response.raise_for_status()

    print(
        f"[OK] Description mise à jour : "
        f"{qualified_name} | Type : {type_name}"
    )

    return True

# ============================================================
# VÉRIFICATION DE LA DESCRIPTION
# ============================================================

def verifier_description(
    qualified_name,
    description_attendue,
    types_entites
):
    entite = rechercher_entite(
        qualified_name,
        types_entites
    )

    if entite is None:
        return False

    description_actuelle = (
        entite
        .get("attributes", {})
        .get("description", "")
    )

    return description_actuelle == description_attendue

# ============================================================
# EXÉCUTION
# ============================================================

def executer_mise_a_jour():
    print("=" * 70)
    print("MISE À JOUR DES RÈGLES MÉTIER DANS APACHE ATLAS")
    print("=" * 70)

    if not verifier_connexion_atlas():
        return

    try:
        types_entites = recuperer_types_entites()
        print(
            f"[OK] {len(types_entites)} types Atlas "
            f"récupérés."
        )

    except requests.exceptions.RequestException as erreur:
        print("[ERREUR] Impossible de récupérer les types Atlas.")
        print(f"Détail : {erreur}")
        return

    nombre_reussites = 0
    nombre_introuvables = 0
    nombre_erreurs = 0

    for qualified_name, description in METADONNEES.items():
        try:
            succes = mettre_a_jour_description(
                qualified_name,
                description,
                types_entites
            )

            if succes:
                nombre_reussites += 1
            else:
                nombre_introuvables += 1

        except requests.exceptions.HTTPError as erreur:
            nombre_erreurs += 1

            code_http = (
                erreur.response.status_code
                if erreur.response is not None
                else "inconnu"
            )

            detail = (
                erreur.response.text
                if erreur.response is not None
                else str(erreur)
            )

            print(
                f"[ERREUR HTTP {code_http}] "
                f"{qualified_name}"
            )
            print(f"Détail : {detail[:500]}")

        except requests.exceptions.RequestException as erreur:
            nombre_erreurs += 1
            print(f"[ERREUR RÉSEAU] {qualified_name}")
            print(f"Détail : {erreur}")

        except Exception as erreur:
            nombre_erreurs += 1
            print(f"[ERREUR] {qualified_name}")
            print(f"Détail : {erreur}")

    print()
    print("=" * 70)
    print("RÉSULTAT")
    print("=" * 70)
    print(f"Entités prévues              : {len(METADONNEES)}")
    print(f"Mises à jour réussies        : {nombre_reussites}")
    print(f"Entités introuvables         : {nombre_introuvables}")
    print(f"Erreurs pendant la mise à jour : {nombre_erreurs}")

# ============================================================
# LANCEMENT DU SCRIPT
# ============================================================

executer_mise_a_jour()

SyntaxError: unterminated string literal (detected at line 39) (2308435375.py, line 39)

In [1]:
import requests
from requests.auth import HTTPBasicAuth

# ============================================================
# CONFIGURATION
# ============================================================

ATLAS_URL = "http://localhost:21000"
AUTH = HTTPBasicAuth("admin", "admin")

HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

# ============================================================
# RÈGLES MÉTIER
# qualifiedName: (type Atlas, description complète)
# ============================================================

REGLES_METIER = {
    "clients@projet_data_lineage": (
        "PostgreSQLTable",
        "Table contenant les informations d'identification des clients. "
        "Règle métier : un client peut posséder plusieurs comptes, tandis "
        "qu'un compte appartient à un seul client."
    ),

    "agences@projet_data_lineage": (
        "PostgreSQLTable",
        "Table contenant les informations relatives aux agences bancaires. "
        "Règle métier : une agence peut gérer plusieurs comptes, tandis "
        "qu'un compte est rattaché à une seule agence."
    ),

    "comptes@projet_data_lineage": (
        "PostgreSQLTable",
        "Table contenant les comptes bancaires. "
        "Règle métier : chaque compte appartient à un client et est rattaché "
        "à une agence. Un compte peut être associé à plusieurs transactions "
        "et participer à plusieurs virements."
    ),

    "transactions@projet_data_lineage": (
        "PostgreSQLTable",
        "Table contenant les opérations enregistrées sur les comptes. "
        "Règle métier : chaque transaction concerne un seul compte, tandis "
        "qu'un compte peut comporter plusieurs transactions."
    ),

    "virements@projet_data_lineage": (
        "PostgreSQLTable",
        "Table contenant les virements réalisés entre les comptes bancaires. "
        "Règle métier : chaque virement possède un compte source et un compte "
        "destinataire. Ces deux comptes doivent exister dans la table comptes."
    ),

    "process_ouverture_compte@projet_data_lineage": (
        "Process",
        "Processus de création d'un compte bancaire à partir des informations "
        "du client et de l'agence. Il utilise les tables clients et agences "
        "comme entrées et produit la table comptes. "
        "Règle métier : le compte produit doit être associé à un client "
        "existant et à une agence existante."
    ),

    "process_comptes_transactions@projet_data_lineage": (
        "Process",
        "Processus reliant les comptes aux transactions. Il utilise la table "
        "comptes comme entrée et produit la table transactions. "
        "Règle métier : chaque transaction produite doit être rattachée à "
        "un compte bancaire existant."
    ),

    "process_execution_virement@projet_data_lineage": (
        "Process",
        "Processus d'exécution d'un virement entre deux comptes bancaires. "
        "Il utilise la table comptes comme entrée et produit la table "
        "virements. Règle métier : chaque virement doit identifier un compte "
        "source et un compte destinataire existants."
    )
}

# ============================================================
# RÉCUPÉRER UNE ENTITÉ PAR SON QUALIFIEDNAME
# ============================================================

def recuperer_entite(type_name, qualified_name):
    url = (
        f"{ATLAS_URL}/api/atlas/v2/entity/"
        f"uniqueAttribute/type/{type_name}"
    )

    response = requests.get(
        url,
        params={"attr:qualifiedName": qualified_name},
        auth=AUTH,
        headers=HEADERS,
        timeout=30
    )

    if response.status_code == 404:
        return None

    response.raise_for_status()
    return response.json().get("entity")

# ============================================================
# METTRE À JOUR LA DESCRIPTION
# ============================================================

def mettre_a_jour_entite(type_name, qualified_name, description):
    entite = recuperer_entite(type_name, qualified_name)

    if entite is None:
        print(f"[INTROUVABLE] {qualified_name} | type : {type_name}")
        return False

    guid = entite.get("guid")
    nom = entite.get("attributes", {}).get(
        "name",
        qualified_name.split("@")[0]
    )

    payload = {
        "entities": [
            {
                "guid": guid,
                "typeName": type_name,
                "attributes": {
                    "name": nom,
                    "qualifiedName": qualified_name,
                    "description": description
                }
            }
        ]
    }

    url = f"{ATLAS_URL}/api/atlas/v2/entity/bulk"

    response = requests.post(
        url,
        json=payload,
        auth=AUTH,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    print(f"[OK] Règle métier ajoutée : {qualified_name}")
    return True

# ============================================================
# EXÉCUTION
# ============================================================

reussites = 0
erreurs = 0

print("=" * 70)
print("AJOUT DES RÈGLES MÉTIER DANS APACHE ATLAS")
print("=" * 70)

for qualified_name, informations in REGLES_METIER.items():
    type_name, description = informations

    try:
        resultat = mettre_a_jour_entite(
            type_name,
            qualified_name,
            description
        )

        if resultat:
            reussites += 1
        else:
            erreurs += 1

    except requests.exceptions.RequestException as erreur:
        erreurs += 1
        print(f"[ERREUR] {qualified_name} : {erreur}")

print("=" * 70)
print(f"Entités prévues : {len(REGLES_METIER)}")
print(f"Mises à jour réussies : {reussites}")
print(f"Entités en erreur : {erreurs}")
print("=" * 70)

AJOUT DES RÈGLES MÉTIER DANS APACHE ATLAS
[ERREUR] clients@projet_data_lineage : 404 Client Error: Not Found for url: http://localhost:21000/api/atlas/v2/entity/bulk
[ERREUR] agences@projet_data_lineage : 404 Client Error: Not Found for url: http://localhost:21000/api/atlas/v2/entity/bulk
[ERREUR] comptes@projet_data_lineage : 404 Client Error: Not Found for url: http://localhost:21000/api/atlas/v2/entity/bulk
[ERREUR] transactions@projet_data_lineage : 404 Client Error: Not Found for url: http://localhost:21000/api/atlas/v2/entity/bulk
[ERREUR] virements@projet_data_lineage : 404 Client Error: Not Found for url: http://localhost:21000/api/atlas/v2/entity/bulk
[OK] Règle métier ajoutée : process_ouverture_compte@projet_data_lineage
[OK] Règle métier ajoutée : process_comptes_transactions@projet_data_lineage
[OK] Règle métier ajoutée : process_execution_virement@projet_data_lineage
Entités prévues : 8
Mises à jour réussies : 3
Entités en erreur : 5


In [1]:
import requests
from requests.auth import HTTPBasicAuth

# ============================================================
# CONFIGURATION APACHE ATLAS
# ============================================================

ATLAS_URL = "http://localhost:21000"
AUTH = HTTPBasicAuth("admin", "admin")

HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

# ============================================================
# RÈGLES MÉTIER DES TABLES
# ============================================================

REGLES_TABLES = {
    "clients@projet_data_lineage": (
        "Table contenant les informations d'identification des clients. "
        "Règle métier : un client peut posséder plusieurs comptes, tandis "
        "qu'un compte appartient à un seul client."
    ),

    "agences@projet_data_lineage": (
        "Table contenant les informations relatives aux agences bancaires. "
        "Règle métier : une agence peut gérer plusieurs comptes, tandis "
        "qu'un compte est rattaché à une seule agence."
    ),

    "comptes@projet_data_lineage": (
        "Table contenant les comptes bancaires. "
        "Règle métier : chaque compte appartient à un client et est rattaché "
        "à une agence. Un compte peut être associé à plusieurs transactions "
        "et participer à plusieurs virements."
    ),

    "transactions@projet_data_lineage": (
        "Table contenant les opérations enregistrées sur les comptes. "
        "Règle métier : chaque transaction concerne un seul compte, tandis "
        "qu'un compte peut comporter plusieurs transactions."
    ),

    "virements@projet_data_lineage": (
        "Table contenant les virements réalisés entre les comptes bancaires. "
        "Règle métier : chaque virement possède un compte source et un compte "
        "destinataire. Ces deux comptes doivent exister dans la table comptes."
    )
}

# ============================================================
# RECHERCHER UNE TABLE PAR QUALIFIEDNAME
# ============================================================

def recuperer_table(qualified_name):
    url = (
        f"{ATLAS_URL}/api/atlas/v2/entity/"
        "uniqueAttribute/type/PostgreSQLTable"
    )

    response = requests.get(
        url,
        params={"attr:qualifiedName": qualified_name},
        auth=AUTH,
        headers=HEADERS,
        timeout=120
    )

    if response.status_code == 404:
        print(f"[INTROUVABLE] {qualified_name}")
        return None

    response.raise_for_status()
    return response.json().get("entity")

# ============================================================
# MODIFIER UNIQUEMENT LA DESCRIPTION
# ============================================================

def modifier_description_table(qualified_name, description):
    table = recuperer_table(qualified_name)

    if table is None:
        return False

    guid = table.get("guid")

    if not guid:
        print(f"[ERREUR] GUID absent : {qualified_name}")
        return False

    url = f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}"

    response = requests.put(
        url,
        params={"name": "description"},
        json=description,
        auth=AUTH,
        headers=HEADERS,
        timeout=120
    )

    if response.status_code not in (200, 201):
        print(
            f"[ERREUR HTTP {response.status_code}] "
            f"{qualified_name}"
        )
        print(response.text[:500])
        return False

    print(f"[OK] Règle métier ajoutée : {qualified_name}")
    return True

# ============================================================
# EXÉCUTION
# ============================================================

reussites = 0
erreurs = 0

print("=" * 70)
print("MISE À JOUR DES RÈGLES MÉTIER DES TABLES")
print("=" * 70)

for qualified_name, description in REGLES_TABLES.items():
    try:
        succes = modifier_description_table(
            qualified_name,
            description
        )

        if succes:
            reussites += 1
        else:
            erreurs += 1

    except requests.exceptions.RequestException as erreur:
        erreurs += 1
        print(f"[ERREUR] {qualified_name} : {erreur}")

print("=" * 70)
print(f"Tables prévues : {len(REGLES_TABLES)}")
print(f"Mises à jour réussies : {reussites}")
print(f"Erreurs : {erreurs}")
print("=" * 70)

MISE À JOUR DES RÈGLES MÉTIER DES TABLES
[OK] Règle métier ajoutée : clients@projet_data_lineage
[OK] Règle métier ajoutée : agences@projet_data_lineage
[OK] Règle métier ajoutée : comptes@projet_data_lineage
[OK] Règle métier ajoutée : transactions@projet_data_lineage
[OK] Règle métier ajoutée : virements@projet_data_lineage
Tables prévues : 5
Mises à jour réussies : 5
Erreurs : 0


In [2]:
import requests
from requests.auth import HTTPBasicAuth

ATLAS_URL = "http://localhost:21000"
AUTH = HTTPBasicAuth("admin", "admin")

HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

def recuperer_definition_type(type_name):
    url = (
        f"{ATLAS_URL}/api/atlas/v2/types/"
        f"entitydef/name/{type_name}"
    )

    response = requests.get(
        url,
        auth=AUTH,
        headers=HEADERS,
        timeout=120
    )

    response.raise_for_status()
    return response.json()


def construire_definition_mise_a_jour(definition):
    attributs = definition.get("attributeDefs", [])

    attribut_existant = any(
        attribut.get("name") == "businessRule"
        for attribut in attributs
    )

    if not attribut_existant:
        attributs.append({
            "name": "businessRule",
            "typeName": "string",
            "isOptional": True,
            "cardinality": "SINGLE",
            "valuesMinCount": 0,
            "valuesMaxCount": 1,
            "isUnique": False,
            "isIndexable": True,
            "includeInNotification": False
        })

    return {
        "category": "ENTITY",
        "name": definition["name"],
        "description": definition.get("description", ""),
        "typeVersion": definition.get("typeVersion", "1.0"),
        "serviceType": definition.get("serviceType"),
        "attributeDefs": attributs,
        "superTypes": definition.get("superTypes", [])
    }


def ajouter_attribut_regle_metier():
    types_a_modifier = [
        "PostgreSQLTable",
        "Process"
    ]

    definitions = []

    for type_name in types_a_modifier:
        definition = recuperer_definition_type(type_name)

        deja_present = any(
            attribut.get("name") == "businessRule"
            for attribut in definition.get("attributeDefs", [])
        )

        if deja_present:
            print(
                f"[DÉJÀ PRÉSENT] businessRule dans {type_name}"
            )
        else:
            print(
                f"[À AJOUTER] businessRule dans {type_name}"
            )

        definitions.append(
            construire_definition_mise_a_jour(definition)
        )

    payload = {
        "entityDefs": definitions
    }

    url = f"{ATLAS_URL}/api/atlas/v2/types/typedefs"

    response = requests.put(
        url,
        json=payload,
        auth=AUTH,
        headers=HEADERS,
        timeout=120
    )

    if response.status_code not in (200, 201):
        print(f"[ERREUR HTTP {response.status_code}]")
        print(response.text[:1000])
        return False

    print("[OK] Attribut businessRule ajouté aux types Atlas.")
    return True


ajouter_attribut_regle_metier()

[À AJOUTER] businessRule dans PostgreSQLTable
[À AJOUTER] businessRule dans Process
[OK] Attribut businessRule ajouté aux types Atlas.


True

In [3]:
METADONNEES_SEPAREES = {
    "clients@projet_data_lineage": {
        "typeName": "PostgreSQLTable",
        "description": (
            "Table contenant les informations d'identification des clients."
        ),
        "businessRule": (
            "Un client peut posséder plusieurs comptes, tandis qu'un compte "
            "appartient à un seul client."
        )
    },

    "agences@projet_data_lineage": {
        "typeName": "PostgreSQLTable",
        "description": (
            "Table contenant les informations relatives aux agences bancaires."
        ),
        "businessRule": (
            "Une agence peut gérer plusieurs comptes, tandis qu'un compte "
            "est rattaché à une seule agence."
        )
    },

    "comptes@projet_data_lineage": {
        "typeName": "PostgreSQLTable",
        "description": (
            "Table contenant les comptes bancaires des clients."
        ),
        "businessRule": (
            "Chaque compte appartient à un client et est rattaché à une "
            "agence. Un compte peut être associé à plusieurs transactions "
            "et participer à plusieurs virements."
        )
    },

    "transactions@projet_data_lineage": {
        "typeName": "PostgreSQLTable",
        "description": (
            "Table contenant les opérations enregistrées sur les comptes."
        ),
        "businessRule": (
            "Chaque transaction concerne un seul compte, tandis qu'un compte "
            "peut comporter plusieurs transactions."
        )
    },

    "virements@projet_data_lineage": {
        "typeName": "PostgreSQLTable",
        "description": (
            "Table contenant les virements réalisés entre les comptes "
            "bancaires."
        ),
        "businessRule": (
            "Chaque virement possède un compte source et un compte "
            "destinataire. Ces deux comptes doivent exister dans la table "
            "comptes."
        )
    },

    "process_ouverture_compte@projet_data_lineage": {
        "typeName": "Process",
        "description": (
            "Processus de création d'un compte bancaire à partir des "
            "informations du client et de l'agence."
        ),
        "businessRule": (
            "Le compte produit doit être associé à un client existant et à "
            "une agence existante."
        )
    },

    "process_comptes_transactions@projet_data_lineage": {
        "typeName": "Process",
        "description": (
            "Processus reliant les comptes aux transactions bancaires."
        ),
        "businessRule": (
            "Chaque transaction produite doit être rattachée à un compte "
            "bancaire existant."
        )
    },

    "process_execution_virement@projet_data_lineage": {
        "typeName": "Process",
        "description": (
            "Processus d'exécution d'un virement entre deux comptes "
            "bancaires."
        ),
        "businessRule": (
            "Chaque virement doit identifier un compte source et un compte "
            "destinataire existants."
        )
    }
}

In [4]:
def recuperer_entite(type_name, qualified_name):
    url = (
        f"{ATLAS_URL}/api/atlas/v2/entity/"
        f"uniqueAttribute/type/{type_name}"
    )

    response = requests.get(
        url,
        params={"attr:qualifiedName": qualified_name},
        auth=AUTH,
        headers=HEADERS,
        timeout=120
    )

    if response.status_code == 404:
        return None

    response.raise_for_status()
    return response.json().get("entity")


def modifier_attribut(guid, nom_attribut, valeur):
    url = f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}"

    response = requests.put(
        url,
        params={"name": nom_attribut},
        json=valeur,
        auth=AUTH,
        headers=HEADERS,
        timeout=120
    )

    if response.status_code not in (200, 201):
        print(
            f"[ERREUR HTTP {response.status_code}] "
            f"Attribut : {nom_attribut}"
        )
        print(response.text[:500])
        return False

    return True


def separer_descriptions_et_regles():
    reussites = 0
    erreurs = 0

    print("=" * 70)
    print("SÉPARATION DES DESCRIPTIONS ET DES RÈGLES MÉTIER")
    print("=" * 70)

    for qualified_name, informations in METADONNEES_SEPAREES.items():
        type_name = informations["typeName"]
        description = informations["description"]
        business_rule = informations["businessRule"]

        try:
            entite = recuperer_entite(
                type_name,
                qualified_name
            )

            if entite is None:
                print(f"[INTROUVABLE] {qualified_name}")
                erreurs += 1
                continue

            guid = entite.get("guid")

            description_ok = modifier_attribut(
                guid,
                "description",
                description
            )

            regle_ok = modifier_attribut(
                guid,
                "businessRule",
                business_rule
            )

            if description_ok and regle_ok:
                print(f"[OK] Séparation réalisée : {qualified_name}")
                reussites += 1
            else:
                erreurs += 1

        except requests.exceptions.RequestException as erreur:
            print(f"[ERREUR] {qualified_name} : {erreur}")
            erreurs += 1

    print("=" * 70)
    print(f"Entités prévues : {len(METADONNEES_SEPAREES)}")
    print(f"Séparations réussies : {reussites}")
    print(f"Erreurs : {erreurs}")
    print("=" * 70)


separer_descriptions_et_regles()

SÉPARATION DES DESCRIPTIONS ET DES RÈGLES MÉTIER
[OK] Séparation réalisée : clients@projet_data_lineage
[OK] Séparation réalisée : agences@projet_data_lineage
[OK] Séparation réalisée : comptes@projet_data_lineage
[OK] Séparation réalisée : transactions@projet_data_lineage
[OK] Séparation réalisée : virements@projet_data_lineage
[OK] Séparation réalisée : process_ouverture_compte@projet_data_lineage
[OK] Séparation réalisée : process_comptes_transactions@projet_data_lineage
[OK] Séparation réalisée : process_execution_virement@projet_data_lineage
Entités prévues : 8
Séparations réussies : 8
Erreurs : 0


In [5]:
qualified_name = "clients@projet_data_lineage"
type_name = "PostgreSQLTable"

entite = recuperer_entite(
    type_name,
    qualified_name
)

if entite:
    attributs = entite.get("attributes", {})

    print("Objet :", attributs.get("name"))
    print("Description :", attributs.get("description"))
    print("Règle métier :", attributs.get("businessRule"))

Objet : clients
Description : Table contenant les informations d'identification des clients.
Règle métier : Un client peut posséder plusieurs comptes, tandis qu'un compte appartient à un seul client.


In [ ]:
import requests
from requests.auth import HTTPBasicAuth

ATLAS_URL = "http://localhost:21000"
AUTH = HTTPBasicAuth("admin", "admin")
HEADERS = {"Accept": "application/json", "Content-Type": "application/json"}
TIMEOUT = 120

PRIMARY_KEYS = [
    "agences.id_agence@projet_data_lineage",
    "clients.id_client@projet_data_lineage",
    "comptes.id_compte@projet_data_lineage",
    "transactions.id_transaction@projet_data_lineage",
    "virements.id_virement@projet_data_lineage",
]


def add_primary_key_attribute():
    type_url = f"{ATLAS_URL}/api/atlas/v2/types/entitydef/name/PostgreSQLColumn"
    response = requests.get(type_url, auth=AUTH, headers=HEADERS, timeout=TIMEOUT)
    response.raise_for_status()
    definition = response.json()

    attributes = definition.get("attributeDefs", [])
    if any(item.get("name") == "isPrimaryKey" for item in attributes):
        print("[OK] L'attribut isPrimaryKey existe déjà.")
        return

    attributes.append({
        "name": "isPrimaryKey",
        "typeName": "boolean",
        "isOptional": True,
        "cardinality": "SINGLE",
        "valuesMinCount": 0,
        "valuesMaxCount": 1,
        "isUnique": False,
        "isIndexable": True,
        "includeInNotification": False,
    })

    updated_definition = {
        "category": "ENTITY",
        "name": definition["name"],
        "description": definition.get("description", ""),
        "typeVersion": definition.get("typeVersion", "1.0"),
        "serviceType": definition.get("serviceType"),
        "attributeDefs": attributes,
        "superTypes": definition.get("superTypes", []),
    }

    update_url = f"{ATLAS_URL}/api/atlas/v2/types/typedefs"
    response = requests.put(
        update_url,
        json={"entityDefs": [updated_definition]},
        auth=AUTH,
        headers=HEADERS,
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    print("[OK] L'attribut isPrimaryKey a été ajouté.")


def get_column(qualified_name):
    url = (
        f"{ATLAS_URL}/api/atlas/v2/entity/"
        "uniqueAttribute/type/PostgreSQLColumn"
    )
    response = requests.get(
        url,
        params={"attr:qualifiedName": qualified_name},
        auth=AUTH,
        headers=HEADERS,
        timeout=TIMEOUT,
    )
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json().get("entity")


def set_primary_key(qualified_name):
    entity = get_column(qualified_name)
    if entity is None:
        print(f"[INTROUVABLE] {qualified_name}")
        return False

    guid = entity.get("guid")
    url = f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}"
    response = requests.put(
        url,
        params={"name": "isPrimaryKey"},
        json=True,
        auth=AUTH,
        headers=HEADERS,
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    print(f"[OK] Clé primaire enregistrée : {qualified_name}")
    return True


def verify_primary_key(qualified_name):
    entity = get_column(qualified_name)
    if entity is None:
        return False
    return entity.get("attributes", {}).get("isPrimaryKey") is True


def main():
    print("=" * 70)
    print("AJOUT DES CLÉS PRIMAIRES DANS APACHE ATLAS")
    print("=" * 70)

    add_primary_key_attribute()

    successes = 0
    errors = 0

    for qualified_name in PRIMARY_KEYS:
        try:
            if set_primary_key(qualified_name):
                successes += 1
            else:
                errors += 1
        except requests.exceptions.RequestException as error:
            errors += 1
            print(f"[ERREUR] {qualified_name} : {error}")

    print("=" * 70)
    print(f"Clés prévues : {len(PRIMARY_KEYS)}")
    print(f"Clés enregistrées : {successes}")
    print(f"Erreurs : {errors}")
    print("=" * 70)

    for qualified_name in PRIMARY_KEYS:
        status = "OUI" if verify_primary_key(qualified_name) else "NON"
        print(f"{qualified_name} | isPrimaryKey = {status}")


if __name__ == "__main__":
    main()


AJOUT DES CLÉS PRIMAIRES DANS APACHE ATLAS
[OK] L'attribut isPrimaryKey a été ajouté.
[OK] Clé primaire enregistrée : agences.id_agence@projet_data_lineage
[OK] Clé primaire enregistrée : clients.id_client@projet_data_lineage
[OK] Clé primaire enregistrée : comptes.id_compte@projet_data_lineage
[OK] Clé primaire enregistrée : transactions.id_transaction@projet_data_lineage
[OK] Clé primaire enregistrée : virements.id_virement@projet_data_lineage
Clés prévues : 5
Clés enregistrées : 5
Erreurs : 0
agences.id_agence@projet_data_lineage | isPrimaryKey = OUI
clients.id_client@projet_data_lineage | isPrimaryKey = OUI
comptes.id_compte@projet_data_lineage | isPrimaryKey = OUI
transactions.id_transaction@projet_data_lineage | isPrimaryKey = OUI
virements.id_virement@projet_data_lineage | isPrimaryKey = OUI


: 

In [1]:
import requests

ATLAS_URL = "http://localhost:21000"
USERNAME = "admin"
PASSWORD = "admin"

# Descriptions métier des colonnes
descriptions = {
    "ID Clients": "Identifiant du client associé à la transaction.",
    "Numéro de compte": "Numéro du compte concerné par la transaction.",
    "Identifiant opération": "Identifiant unique de l'opération enregistrée.",
    "Type de transaction": "Nature ou catégorie de l'opération enregistrée.",
    "Status opération": "Statut de traitement de l'opération.",
    "Date": "Date associée à l'opération.",
    "Montant": "Valeur monétaire associée à la transaction."
}


def rechercher_colonne(nom_colonne):
    """
    Recherche une colonne dans Apache Atlas à partir de son nom.
    Adapte SQLiteColumn si ton type Atlas porte un autre nom.
    """

    url = f"{ATLAS_URL}/api/atlas/v2/search/basic"

    params = {
        "typeName": "SQLiteColumn",
        "query": nom_colonne
    }

    response = requests.get(
        url,
        params=params,
        auth=(USERNAME, PASSWORD)
    )

    response.raise_for_status()

    entities = response.json().get("entities", [])

    # On garde uniquement la colonne de la table transactions
    for entity in entities:
        attrs = entity.get("attributes", {})

        name = attrs.get("name", "")
        qualified_name = attrs.get("qualifiedName", "")

        if (
            name.lower() == nom_colonne.lower()
            and "transactions" in qualified_name.lower()
        ):
            return entity

    return None


def modifier_description(guid, description):
    """
    Met à jour uniquement l'attribut description de l'entité Atlas.
    """

    url = f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}"

    params = {
        "name": "description"
    }

    response = requests.put(
        url,
        params=params,
        json=description,
        auth=(USERNAME, PASSWORD),
        headers={"Content-Type": "application/json"}
    )

    if response.status_code in [200, 204]:
        return True

    print(
        f"Erreur Atlas {response.status_code} : "
        f"{response.text}"
    )

    return False


for nom_colonne, description in descriptions.items():

    print(f"\nRecherche : {nom_colonne}")

    colonne = rechercher_colonne(nom_colonne)

    if not colonne:
        print(f"❌ Colonne introuvable : {nom_colonne}")
        continue

    guid = colonne["guid"]

    print(f"GUID trouvé : {guid}")

    if modifier_description(guid, description):
        print(
            f"✅ Description ajoutée à {nom_colonne} : "
            f"{description}"
        )


Recherche : ID Clients
GUID trouvé : 8e29f74b-d26e-4e5c-bf6b-61bec1062cf5
✅ Description ajoutée à ID Clients : Identifiant du client associé à la transaction.

Recherche : Numéro de compte
GUID trouvé : c1c7a2d3-4056-436e-94aa-df167c27f51e
✅ Description ajoutée à Numéro de compte : Numéro du compte concerné par la transaction.

Recherche : Identifiant opération
GUID trouvé : 89b5fd56-86b2-4c01-956a-0c391ac1125f
✅ Description ajoutée à Identifiant opération : Identifiant unique de l'opération enregistrée.

Recherche : Type de transaction
GUID trouvé : caf5b7fe-c33e-4f1a-b2dd-f082573827ed
✅ Description ajoutée à Type de transaction : Nature ou catégorie de l'opération enregistrée.

Recherche : Status opération
GUID trouvé : 4b120b87-adf2-47c2-aeb6-ac7b7979278f
✅ Description ajoutée à Status opération : Statut de traitement de l'opération.

Recherche : Date
GUID trouvé : 2f5ca54d-e332-4545-89fc-a0b0e36902ca
✅ Description ajoutée à Date : Date associée à l'opération.

Recherche : Montant


In [1]:
import requests
from requests.auth import HTTPBasicAuth
import json

ATLAS_URL = "http://localhost:21000"
AUTH = HTTPBasicAuth("admin", "admin")

url = f"{ATLAS_URL}/api/atlas/v2/types/typedef/name/PostgreSQLTable"

response = requests.get(url, auth=AUTH)

print("Status :", response.status_code)

if response.ok:
    print(json.dumps(response.json(), indent=4, ensure_ascii=False))
else:
    print(response.text)

Status : 200
{
    "category": "ENTITY",
    "guid": "ef174d8c-25cc-4c2b-9698-095be3dbe6b5",
    "createdBy": "admin",
    "updatedBy": "admin",
    "createTime": 1788252432523,
    "updateTime": 1789027287948,
    "version": 2,
    "name": "PostgreSQLTable",
    "description": "Table d'une base PostgreSQL",
    "typeVersion": "1.0",
    "serviceType": "PostgreSQL",
    "attributeDefs": [
        {
            "name": "databaseName",
            "typeName": "string",
            "isOptional": true,
            "cardinality": "SINGLE",
            "valuesMinCount": 0,
            "valuesMaxCount": 1,
            "isUnique": false,
            "isIndexable": true,
            "includeInNotification": false
        },
        {
            "name": "businessRule",
            "typeName": "string",
            "isOptional": true,
            "cardinality": "SINGLE",
            "valuesMinCount": 0,
            "valuesMaxCount": 1,
            "isUnique": false,
            "isIndexable": tr

In [2]:
import requests
from requests.auth import HTTPBasicAuth

ATLAS_URL = "http://localhost:21000"
AUTH = HTTPBasicAuth("admin", "admin")

response = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/types/typedef/name/PostgreSQLTable",
    auth=AUTH
)

data = response.json()

print("Attributs de PostgreSQLTable :")

for attr in data.get("attributeDefs", []):
    print(
        "-",
        attr.get("name"),
        "| type =", attr.get("typeName"),
        "| cardinalité =", attr.get("cardinality")
    )

print("\nRelations :")

for rel in data.get("relationshipAttributeDefs", []):
    print(
        "-",
        rel.get("name"),
        "| type =", rel.get("typeName")
    )

Attributs de PostgreSQLTable :
- databaseName | type = string | cardinalité = SINGLE
- businessRule | type = string | cardinalité = SINGLE

Relations :
- schema | type = array<avro_schema>
- inputToProcesses | type = array<Process>
- database | type = PostgreSQLDatabase
- columns | type = array<PostgreSQLColumn>
- meanings | type = array<AtlasGlossaryTerm>
- outputFromProcesses | type = array<Process>


In [3]:
import requests
from requests.auth import HTTPBasicAuth

ATLAS_URL = "http://localhost:21000"
AUTH = HTTPBasicAuth("admin", "admin")

# =========================================================
# 1. Rechercher la table comptes
# =========================================================

params = {
    "typeName": "PostgreSQLTable",
    "attrName": "name",
    "attrValuePrefix": "comptes"
}

response = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/search/attribute",
    params=params,
    auth=AUTH
)

print("Status :", response.status_code)

data = response.json()

for entity in data.get("entities", []):
    print(
        entity.get("guid"),
        "|",
        entity.get("typeName"),
        "|",
        entity.get("displayText")
    )

Status : 200
4c832ff2-a6c6-428a-8398-c6fc8235c3e4 | PostgreSQLTable | comptes


In [4]:
import requests
from requests.auth import HTTPBasicAuth

ATLAS_URL = "http://localhost:21000"
AUTH = HTTPBasicAuth("admin", "admin")

GUID_COMPTES = "4c832ff2-a6c6-428a-8398-c6fc8235c3e4"

response = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/entity/guid/{GUID_COMPTES}",
    auth=AUTH
)

print("Status :", response.status_code)

data = response.json()
entity = data["entity"]

print("\nTable :", entity["attributes"].get("name"))
print("QualifiedName :", entity["attributes"].get("qualifiedName"))

columns = (
    entity
    .get("relationshipAttributes", {})
    .get("columns", [])
)

print("\nColonnes rattachées à comptes :")

if not columns:
    print("Aucune colonne actuellement rattachée à comptes.")
else:
    for column in columns:
        print(
            "-",
            column.get("displayText"),
            "| type =", column.get("typeName"),
            "| GUID =", column.get("guid")
        )

Status : 200

Table : comptes
QualifiedName : comptes@projet_data_lineage

Colonnes rattachées à comptes :
- solde | type = PostgreSQLColumn | GUID = bf1ad450-ceae-4907-b828-7dcd11dbb3f3
- id_client | type = PostgreSQLColumn | GUID = 9717208b-282e-4b9a-abcf-c96c1823e0c7
- id_compte | type = PostgreSQLColumn | GUID = 021640e4-152c-42df-bc6e-cbcbe21daf4c
- id_agence | type = PostgreSQLColumn | GUID = 689297a2-ba77-4ffa-a1fe-74dfad52f8b6
- type_compte | type = PostgreSQLColumn | GUID = 17761509-ded4-4d9b-94d6-d73c81ed5259


In [1]:
import requests
import pandas as pd

ATLAS_URL = "http://localhost:21000"
ATLAS_AUTH = ("admin", "admin")  # adapte si nécessaire

# Recherche de toutes les entités hdfs_path
url = f"{ATLAS_URL}/api/atlas/v2/search/basic"

params = {
    "typeName": "hdfs_path",
    "limit": 1000
}

r = requests.get(
    url,
    params=params,
    auth=ATLAS_AUTH,
    timeout=30
)

r.raise_for_status()

entities = r.json().get("entities", [])

print("Nombre d'entités hdfs_path :", len(entities))

Nombre d'entités hdfs_path : 1


In [3]:
for e in entities:
    print("GUID :", e.get("guid"))
    print("Name :", e.get("displayText"))
    print("Type :", e.get("typeName"))
    print("Status :", e.get("status"))
    print("Attributes :", e.get("attributes"))
    print("-" * 60)

GUID : ca02c6b9-c482-4609-8c0d-64d3f0c6511f
Name : test.txt
Type : hdfs_path
Status : ACTIVE
Attributes : {'createTime': 0, 'qualifiedName': 'test.txt@data_lineage', 'name': 'test.txt'}
------------------------------------------------------------


In [4]:
import requests

params = {
    "typeName": "hdfs_path",
    "limit": 1000,
    "excludeDeletedEntities": False
}

r = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/search/basic",
    params=params,
    auth=ATLAS_AUTH,
    timeout=30
)

r.raise_for_status()

all_entities = r.json().get("entities", [])

print("Nombre total :", len(all_entities))
print()

for e in all_entities:
    attrs = e.get("attributes", {}) or {}

    print("GUID          :", e.get("guid"))
    print("Name          :", attrs.get("name") or e.get("displayText"))
    print("QualifiedName :", attrs.get("qualifiedName"))
    print("Status        :", e.get("status"))
    print("-" * 70)

Nombre total : 1

GUID          : ca02c6b9-c482-4609-8c0d-64d3f0c6511f
Name          : test.txt
QualifiedName : test.txt@data_lineage
Status        : ACTIVE
----------------------------------------------------------------------


In [5]:
def search_atlas_name(name):
    params = {
        "query": name,
        "limit": 100
    }

    r = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/search/basic",
        params=params,
        auth=ATLAS_AUTH,
        timeout=30
    )

    r.raise_for_status()
    return r.json().get("entities", [])


for name in ["transactions_sep.xlsx", "transactions1.csv"]:

    print("\nRECHERCHE :", name)

    results = search_atlas_name(name)

    print("Résultats :", len(results))

    for e in results:
        attrs = e.get("attributes", {}) or {}

        print(
            e.get("guid"),
            "|",
            e.get("typeName"),
            "|",
            e.get("status"),
            "|",
            attrs.get("qualifiedName")
        )


RECHERCHE : transactions_sep.xlsx
Résultats : 0

RECHERCHE : transactions1.csv
Résultats : 3
8198e80f-3bf7-4178-b650-b68d9c15688d | Process | ACTIVE | Chargement_SQLite@projet_data_lineage
726ffa2d-e152-49d5-9c60-4b07c5a74486 | SQLiteDatabase | ACTIVE | transactions1.db@projet_data_lineage
49993d0f-8351-4bc6-8dc2-aca053bfe465 | SQLiteTable | ACTIVE | transactions@transactions1.db@projet_data_lineage


In [6]:
import requests

ATLAS_URL = "http://localhost:21000"
ATLAS_AUTH = ("admin", "admin")


def search_process(process_name):
    params = {
        "typeName": "Process",
        "query": process_name,
        "limit": 100
    }

    r = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/search/basic",
        params=params,
        auth=ATLAS_AUTH,
        timeout=30
    )

    r.raise_for_status()

    return r.json().get("entities", [])


for process_name in [
    "Preparation_transactions",
    "Chargement_SQLite"
]:
    print("\n" + "=" * 80)
    print("PROCESSUS :", process_name)

    results = search_process(process_name)

    for e in results:
        attrs = e.get("attributes", {}) or {}

        print("GUID          :", e.get("guid"))
        print("Name          :", attrs.get("name") or e.get("displayText"))
        print("QualifiedName :", attrs.get("qualifiedName"))
        print("Status        :", e.get("status"))


PROCESSUS : Preparation_transactions
GUID          : aee1daa1-095c-43ea-a9ef-4d7c9067afa1
Name          : Preparation_transactions
QualifiedName : Preparation_transactions@projet_data_lineage
Status        : ACTIVE

PROCESSUS : Chargement_SQLite
GUID          : 8198e80f-3bf7-4178-b650-b68d9c15688d
Name          : Chargement_SQLite
QualifiedName : Chargement_SQLite@projet_data_lineage
Status        : ACTIVE


In [7]:
def get_entity_complete(guid):

    r = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}",
        auth=ATLAS_AUTH,
        timeout=30
    )

    r.raise_for_status()

    return r.json()


process_names = [
    "Preparation_transactions",
    "Chargement_SQLite"
]

process_guids = {}

for process_name in process_names:

    results = search_process(process_name)

    exact = [
        e for e in results
        if (
            (e.get("attributes", {}) or {}).get("name")
            or e.get("displayText")
        ) == process_name
    ]

    if exact:
        process_guids[process_name] = exact[0]["guid"]


print(process_guids)

{'Preparation_transactions': 'aee1daa1-095c-43ea-a9ef-4d7c9067afa1', 'Chargement_SQLite': '8198e80f-3bf7-4178-b650-b68d9c15688d'}


In [8]:
def get_entity_complete(guid):
    r = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}",
        auth=ATLAS_AUTH,
        timeout=30
    )
    r.raise_for_status()
    return r.json()


process_guids = {
    "Preparation_transactions":
        "aee1daa1-095c-43ea-a9ef-4d7c9067afa1",

    "Chargement_SQLite":
        "8198e80f-3bf7-4178-b650-b68d9c15688d"
}


for process_name, guid in process_guids.items():

    print("\n" + "=" * 80)
    print("PROCESSUS :", process_name)

    data = get_entity_complete(guid)
    entity = data.get("entity", {})

    relationships = entity.get("relationshipAttributes", {}) or {}

    print("\nINPUTS :")

    inputs = relationships.get("inputs", []) or []

    if not inputs:
        print("Aucun input")

    for x in inputs:
        print("GUID :", x.get("guid"))
        print("Type :", x.get("typeName"))
        print("Nom  :", x.get("displayText"))
        print("-" * 40)

    print("\nOUTPUTS :")

    outputs = relationships.get("outputs", []) or []

    if not outputs:
        print("Aucun output")

    for x in outputs:
        print("GUID :", x.get("guid"))
        print("Type :", x.get("typeName"))
        print("Nom  :", x.get("displayText"))
        print("-" * 40)


PROCESSUS : Preparation_transactions

INPUTS :
GUID : 25ebaecf-0ac6-4adc-91c4-8b54d87c50ea
Type : hdfs_path
Nom  : transactions_sep.xlsx
----------------------------------------

OUTPUTS :
GUID : d40ed547-2948-4b3a-b1c8-077b0bd7cbc6
Type : hdfs_path
Nom  : transactions1.csv
----------------------------------------

PROCESSUS : Chargement_SQLite

INPUTS :
GUID : d40ed547-2948-4b3a-b1c8-077b0bd7cbc6
Type : hdfs_path
Nom  : transactions1.csv
----------------------------------------

OUTPUTS :
GUID : 49993d0f-8351-4bc6-8dc2-aca053bfe465
Type : SQLiteTable
Nom  : transactions
----------------------------------------


In [9]:
file_guids = {
    "transactions_sep.xlsx":
        "25ebaecf-0ac6-4adc-91c4-8b54d87c50ea",

    "transactions1.csv":
        "d40ed547-2948-4b3a-b1c8-077b0bd7cbc6"
}


for name, guid in file_guids.items():

    print("\n" + "=" * 80)
    print("ENTITE :", name)

    try:
        data = get_entity_complete(guid)
        entity = data.get("entity", {})

        attrs = entity.get("attributes", {}) or {}

        print("GUID          :", entity.get("guid"))
        print("Status        :", entity.get("status"))
        print("Type          :", entity.get("typeName"))
        print("Name          :", attrs.get("name"))
        print("QualifiedName :", attrs.get("qualifiedName"))
        print("Path          :", attrs.get("path"))

    except Exception as e:
        print("ERREUR :", e)


ENTITE : transactions_sep.xlsx
GUID          : 25ebaecf-0ac6-4adc-91c4-8b54d87c50ea
Status        : ACTIVE
Type          : hdfs_path
Name          : transactions_sep.xlsx
QualifiedName : transactions_sep.xlsx@projet_data_lineage
Path          : /data/transactions_sep.xlsx

ENTITE : transactions1.csv
GUID          : d40ed547-2948-4b3a-b1c8-077b0bd7cbc6
Status        : ACTIVE
Type          : hdfs_path
Name          : transactions1.csv
QualifiedName : transactions1.csv@projet_data_lineage
Path          : /data/transactions1.csv


In [10]:
import requests
import pandas as pd

ATLAS_URL = "http://localhost:21000"
ATLAS_AUTH = ("admin", "admin")

In [11]:
url = f"{ATLAS_URL}/api/atlas/v2/search/dsl"

params = {
    "query": "from hdfs_path",
    "limit": 1000
}

r = requests.get(
    url,
    params=params,
    auth=ATLAS_AUTH,
    timeout=30
)

print("HTTP :", r.status_code)

r.raise_for_status()

data = r.json()

advanced_entities = data.get("entities", [])

print("Nombre d'entités retournées :", len(advanced_entities))


HTTP : 200
Nombre d'entités retournées : 8


In [12]:
rows = []

for e in advanced_entities:
    attrs = e.get("attributes", {}) or {}

    rows.append({
        "GUID": e.get("guid"),
        "Name": attrs.get("name") or e.get("displayText"),
        "QualifiedName": attrs.get("qualifiedName"),
        "Type": e.get("typeName"),
        "Status": e.get("status")
    })

df_advanced = pd.DataFrame(rows)

df_advanced

,GUID,Name,QualifiedName,Type,Status
0,25ebaecf-0ac6-4adc-91c4-8b54d87c50ea,transactions_sep.xlsx,transactions_sep.xlsx@projet_data_lineage,hdfs_path,ACTIVE
1,d40ed547-2948-4b3a-b1c8-077b0bd7cbc6,transactions1.csv,transactions1.csv@projet_data_lineage,hdfs_path,ACTIVE
2,dd94ffa8-34a8-4014-84df-d4254675ff18,transactions_sep.xlsx,transactions_sep.xlsx@projet_data_lineage,hdfs_path,ACTIVE
3,96585397-851b-4b37-a610-c7687437aafa,transactions_sep.xlsx,transactions_sep.xlsx@projet_data_lineage,hdfs_path,ACTIVE
4,6aa3c8c5-963e-4126-927e-13782181ac79,test.txt,test.txt@projet_tracking,hdfs_path,ACTIVE
5,a89579cb-7b6e-4f83-935a-6a5bbe237710,transactions_sep.xlsx,transactions_sep.xlsx@projet_data_lineage,hdfs_path,ACTIVE
6,c4c306e8-5c66-46af-9a21-cbbe6de03546,transactions_sep.xlsx,transactions_sep.xlsx@projet_data_lineage,hdfs_path,ACTIVE
7,ca02c6b9-c482-4609-8c0d-64d3f0c6511f,test.txt,test.txt@data_lineage,hdfs_path,ACTIVE


In [13]:
duplicate_guids = [
    "dd94ffa8-34a8-4014-84df-d4254675ff18",
    "96585397-851b-4b37-a610-c7687437aafa",
    "a89579cb-7b6e-4f83-935a-6a5bbe237710",
    "c4c306e8-5c66-46af-9a21-cbbe6de03546"
]

for guid in duplicate_guids:

    print("\n" + "=" * 80)
    print("GUID :", guid)

    # Entité
    r = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}",
        auth=ATLAS_AUTH,
        timeout=30
    )

    print("Entity HTTP :", r.status_code)

    if r.ok:
        entity = r.json().get("entity", {})
        attrs = entity.get("attributes", {}) or {}

        print("Name :", attrs.get("name"))
        print("QualifiedName :", attrs.get("qualifiedName"))
        print("Status :", entity.get("status"))

    # Lineage
    r2 = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/lineage/{guid}",
        params={
            "direction": "BOTH",
            "depth": 10
        },
        auth=ATLAS_AUTH,
        timeout=30
    )

    print("Lineage HTTP :", r2.status_code)

    if r2.ok:
        lineage = r2.json()
        relations = lineage.get("relations", []) or []

        print("Nombre de relations :", len(relations))

        for relation in relations:
            print(
                relation.get("fromEntityId"),
                "→",
                relation.get("toEntityId")
            )


GUID : dd94ffa8-34a8-4014-84df-d4254675ff18
Entity HTTP : 200
Name : transactions_sep.xlsx
QualifiedName : transactions_sep.xlsx@projet_data_lineage
Status : ACTIVE
Lineage HTTP : 200
Nombre de relations : 0

GUID : 96585397-851b-4b37-a610-c7687437aafa
Entity HTTP : 200
Name : transactions_sep.xlsx
QualifiedName : transactions_sep.xlsx@projet_data_lineage
Status : ACTIVE
Lineage HTTP : 200
Nombre de relations : 0

GUID : a89579cb-7b6e-4f83-935a-6a5bbe237710
Entity HTTP : 200
Name : transactions_sep.xlsx
QualifiedName : transactions_sep.xlsx@projet_data_lineage
Status : ACTIVE
Lineage HTTP : 200
Nombre de relations : 0

GUID : c4c306e8-5c66-46af-9a21-cbbe6de03546
Entity HTTP : 200
Name : transactions_sep.xlsx
QualifiedName : transactions_sep.xlsx@projet_data_lineage
Status : ACTIVE
Lineage HTTP : 200
Nombre de relations : 0


In [14]:
all_transactions_sep = [
    "25ebaecf-0ac6-4adc-91c4-8b54d87c50ea",  # attendu : bon
    "dd94ffa8-34a8-4014-84df-d4254675ff18",
    "96585397-851b-4b37-a610-c7687437aafa",
    "a89579cb-7b6e-4f83-935a-6a5bbe237710",
    "c4c306e8-5c66-46af-9a21-cbbe6de03546"
]

for guid in all_transactions_sep:

    r = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/lineage/{guid}",
        params={"direction": "BOTH", "depth": 10},
        auth=ATLAS_AUTH,
        timeout=30
    )

    r.raise_for_status()

    relations = r.json().get("relations", []) or []

    print(
        guid,
        "→",
        len(relations),
        "relation(s)"
    )

25ebaecf-0ac6-4adc-91c4-8b54d87c50ea → 4 relation(s)
dd94ffa8-34a8-4014-84df-d4254675ff18 → 0 relation(s)
96585397-851b-4b37-a610-c7687437aafa → 0 relation(s)
a89579cb-7b6e-4f83-935a-6a5bbe237710 → 0 relation(s)
c4c306e8-5c66-46af-9a21-cbbe6de03546 → 0 relation(s)


In [15]:
duplicate_guids = [
    "dd94ffa8-34a8-4014-84df-d4254675ff18",
    "96585397-851b-4b37-a610-c7687437aafa",
    "a89579cb-7b6e-4f83-935a-6a5bbe237710",
    "c4c306e8-5c66-46af-9a21-cbbe6de03546"
]

GOOD_GUID = "25ebaecf-0ac6-4adc-91c4-8b54d87c50ea"

print("GUID conservé :", GOOD_GUID)
print("GUID à supprimer :", len(duplicate_guids))

GUID conservé : 25ebaecf-0ac6-4adc-91c4-8b54d87c50ea
GUID à supprimer : 4


In [16]:
for guid in duplicate_guids:

    r = requests.delete(
        f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}",
        auth=ATLAS_AUTH,
        timeout=30
    )

    print(
        guid,
        "→ HTTP",
        r.status_code
    )

    if not r.ok:
        print("Erreur :", r.text)

dd94ffa8-34a8-4014-84df-d4254675ff18 → HTTP 200
96585397-851b-4b37-a610-c7687437aafa → HTTP 200
a89579cb-7b6e-4f83-935a-6a5bbe237710 → HTTP 200
c4c306e8-5c66-46af-9a21-cbbe6de03546 → HTTP 200


In [17]:
for guid in [GOOD_GUID] + duplicate_guids:

    r = requests.get(
        f"{ATLAS_URL}/api/atlas/v2/entity/guid/{guid}",
        auth=ATLAS_AUTH,
        timeout=30
    )

    if r.ok:
        entity = r.json().get("entity", {})

        print(
            guid,
            "→",
            entity.get("status")
        )
    else:
        print(
            guid,
            "→ non retourné, HTTP",
            r.status_code
        )

25ebaecf-0ac6-4adc-91c4-8b54d87c50ea → ACTIVE
dd94ffa8-34a8-4014-84df-d4254675ff18 → DELETED
96585397-851b-4b37-a610-c7687437aafa → DELETED
a89579cb-7b6e-4f83-935a-6a5bbe237710 → DELETED
c4c306e8-5c66-46af-9a21-cbbe6de03546 → DELETED


In [18]:
r = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/lineage/{GOOD_GUID}",
    params={
        "direction": "DOWNSTREAM",
        "depth": 10
    },
    auth=ATLAS_AUTH,
    timeout=30
)

r.raise_for_status()

lineage = r.json()

print("Relations :", len(lineage.get("relations", []) or []))

for guid, entity in (lineage.get("guidEntityMap", {}) or {}).items():
    print(
        guid,
        "→",
        entity.get("displayText"),
        "(" + str(entity.get("typeName")) + ")"
    )

HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:21000/api/atlas/v2/lineage/25ebaecf-0ac6-4adc-91c4-8b54d87c50ea?direction=DOWNSTREAM&depth=10

In [19]:
GOOD_GUID = "25ebaecf-0ac6-4adc-91c4-8b54d87c50ea"

url = f"{ATLAS_URL}/api/atlas/v2/lineage/{GOOD_GUID}"

params = {
    "direction": "BOTH",
    "depth": 10
}

r = requests.get(
    url,
    params=params,
    auth=ATLAS_AUTH,
    timeout=30
)

print("URL :", r.url)
print("HTTP :", r.status_code)
print("Réponse Atlas :")
print(r.text)

URL : http://localhost:21000/api/atlas/v2/lineage/25ebaecf-0ac6-4adc-91c4-8b54d87c50ea?direction=BOTH&depth=10
HTTP : 200
Réponse Atlas :
{"baseEntityGuid":"25ebaecf-0ac6-4adc-91c4-8b54d87c50ea","lineageDirection":"BOTH","lineageDepth":10,"guidEntityMap":{"25ebaecf-0ac6-4adc-91c4-8b54d87c50ea":{"typeName":"hdfs_path","attributes":{"createTime":1786316400000,"qualifiedName":"transactions_sep.xlsx@projet_data_lineage","name":"transactions_sep.xlsx","description":"Fichier source contenant les données initiales des transactions du cas d’usage pilote"},"guid":"25ebaecf-0ac6-4adc-91c4-8b54d87c50ea","status":"ACTIVE","displayText":"transactions_sep.xlsx","classificationNames":[],"meaningNames":[],"meanings":[]},"aee1daa1-095c-43ea-a9ef-4d7c9067afa1":{"typeName":"Process","attributes":{"qualifiedName":"Preparation_transactions@projet_data_lineage","name":"Preparation_transactions","description":"Processus de préparation réalisé avec Python/Pandas : suppression des colonnes Localisation et Targ

In [20]:
PROCESS_GUID = "aee1daa1-095c-43ea-a9ef-4d7c9067afa1"

r_process = requests.get(
    f"{ATLAS_URL}/api/atlas/v2/entity/guid/{PROCESS_GUID}",
    auth=ATLAS_AUTH,
    timeout=30
)

print("PROCESS HTTP :", r_process.status_code)

if r_process.ok:
    process = r_process.json()["entity"]

    relationships = process.get("relationshipAttributes", {})

    print("\nINPUTS :")
    for x in relationships.get("inputs", []) or []:
        print(
            x.get("displayText"),
            "|",
            x.get("guid"),
            "|",
            x.get("entityStatus")
        )

    print("\nOUTPUTS :")
    for x in relationships.get("outputs", []) or []:
        print(
            x.get("displayText"),
            "|",
            x.get("guid"),
            "|",
            x.get("entityStatus")
        )
else:
    print(r_process.text)

PROCESS HTTP : 200

INPUTS :
transactions_sep.xlsx | 25ebaecf-0ac6-4adc-91c4-8b54d87c50ea | ACTIVE

OUTPUTS :
transactions1.csv | d40ed547-2948-4b3a-b1c8-077b0bd7cbc6 | ACTIVE


In [ ]:
@scoped_cache(st.cache_data(ttl=30, show_spinner=False))
def search_type_advanced(type_name):
    """
    Recherche toutes les entités d'un type avec la recherche DSL Atlas.
    """

    data = atlas_get(
        "/api/atlas/v2/search/dsl",
        {
            "query": f"from {type_name}",
            "limit": 1000
        }
    )

    return data.get("entities", [])

NameError: name 'scoped_cache' is not defined

: 